# YOLOv8m DEHB: Cross Year

This notebook implements the cross year DEHB experiment for YOLOv8m. Hyperparameter optimization uses only the source year training and validation data. The target year test split remains isolated until the best full fidelity configuration has been frozen.

The search optimizes the same five parameters used by the companion Random Search and Hyperband notebooks. DEHB uses 20 and 40 epoch fidelities across two iterations for exactly 320 HPO epoch units. The selected 40 epoch configuration is then retrained independently with seeds 42, 123, and 999 for the separate final evaluation.

The notebook is designed for reproducible GitHub use. Local paths and personal identifiers are not written to saved manifests or result tables. Large temporary YOLO data and HPO trial folders are removed only after the complete workflow succeeds.


## Module 1: Imports and experiment configuration

Defines the cross year direction, portable project paths, target classes, split protocol, YOLO runtime settings, five parameter search space, DEHB controls, seeds, and exact compute budgets. No dataset files are read in this module.


In [ ]:
from pathlib import Path
from datetime import datetime
from statistics import mean, stdev
from typing import Dict, Any, List, Optional, Tuple
from collections import Counter, defaultdict

import csv
import json
import math
import os
import random
import shutil
import sys
import time
import uuid
import re
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO


# ============================================================
# A. Cross-year direction
# ============================================================
# Change only these two values to reverse the experiment.
TRAIN_YEAR = "2021"
TEST_YEAR = "2022"

VALID_YEARS = {"2021", "2022"}
if TRAIN_YEAR not in VALID_YEARS or TEST_YEAR not in VALID_YEARS:
    raise ValueError("TRAIN_YEAR and TEST_YEAR must each be '2021' or '2022'.")
if TRAIN_YEAR == TEST_YEAR:
    raise ValueError("Cross-year evaluation requires TRAIN_YEAR != TEST_YEAR.")

TRAIN_DATASET_TAG = f"Year{TRAIN_YEAR}"
TEST_DATASET_TAG = f"Year{TEST_YEAR}"
CROSS_YEAR_TAG = f"Y{TRAIN_YEAR}_to_Y{TEST_YEAR}"


# ============================================================
# B. Portable paths
# ============================================================
# Expected dataset locations:
# ./Datasets/Year2021
# ./Datasets/Year2022
WORKSPACE_ROOT = Path.cwd()
DATASETS_ROOT = WORKSPACE_ROOT / "Datasets"
TRAIN_SOURCE_ROOT = DATASETS_ROOT / TRAIN_DATASET_TAG
TEST_TARGET_ROOT = DATASETS_ROOT / TEST_DATASET_TAG

PROJECT_ROOT = (
    WORKSPACE_ROOT
    / "outputs"
    / f"yolov8m_dehb_cross_year_{CROSS_YEAR_TAG.lower()}"
)
RUNS_ROOT = PROJECT_ROOT / "runs"
SPLITS_ROOT = PROJECT_ROOT / "splits"
STATS_ROOT = PROJECT_ROOT / "stats"
YOLO_DATA_ROOT = PROJECT_ROOT / "yolo_data"


# ============================================================
# C. Classes and label aliases
# ============================================================
CLASS_NAMES = [
    "Waterhemp",
    "Carpetweed",
    "Morningglory",
    "Goosegrass",
    "SpottedSpurge",
    "PalmerAmaranth",
    "Purslane",
    "Ragweed",
]
TARGET_CLASS_SET = set(CLASS_NAMES)

LABEL_ALIASES = {
    "morninglory": "Morningglory",
    "morning glory": "Morningglory",
    "morningglory": "Morningglory",
    "spotted spurge": "SpottedSpurge",
    "spottedspurge": "SpottedSpurge",
    "palmer amaranth": "PalmerAmaranth",
    "palmeramaranth": "PalmerAmaranth",
}


# ============================================================
# D. Split and seed protocol
# ============================================================
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

OPTIMIZATION_SPLIT_SEED = 42
OPTIMIZATION_TRAIN_SEED = 42
FINAL_EVALUATION_SEEDS = [42, 123, 999]


# ============================================================
# E. Year2021 source-training subsampling
# ============================================================
SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE = True
SUBSAMPLE_TARGET_MODE = "target_train_size"
SUBSAMPLE_SEED_OFFSET = 10000


# ============================================================
# F. Shared YOLO runtime protocol
# ============================================================
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 8
PATIENCE = 0
IMAGE_PLACEMENT_MODE = "symlink"
CACHE_MODE = "disk"

SCREEN_EPOCHS = 20
SCREEN_WARMUP_EPOCHS = 1.5
SCREEN_CLOSE_MOSAIC = 5

FULL_EPOCHS = 40
FULL_WARMUP_EPOCHS = 3.0
FULL_CLOSE_MOSAIC = 10

FINAL_EPOCHS = 40
FINAL_WARMUP_EPOCHS = 3.0
FINAL_CLOSE_MOSAIC = 10


# ============================================================
# G. Focused 5D HPO search space
# ============================================================
SEARCH_SPACE_VERSION = "Focused5D_CrossYear_DEHB20to40_2Iterations_mAP5095_v1"

SEARCH_SPACE = {
    "learning_rate": {"low": 0.0010, "high": 0.0100, "distribution": "log"},
    "lrf":           {"low": 0.0100, "high": 0.2000, "distribution": "log"},
    "momentum":      {"low": 0.8500, "high": 0.9700, "distribution": "linear"},
    "weight_decay":  {"low": 0.0001, "high": 0.0010, "distribution": "log"},
    "scale":         {"low": 0.2000, "high": 0.8000, "distribution": "linear"},
}
SEARCH_PARAMETER_ORDER = [
    "learning_rate",
    "lrf",
    "momentum",
    "weight_decay",
    "scale",
]

REFERENCE_UNTUNED_VALUES = {
    "learning_rate": 0.010,
    "lrf": 0.010,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "scale": 0.50,
}

for parameter_name, reference_value in REFERENCE_UNTUNED_VALUES.items():
    spec = SEARCH_SPACE[parameter_name]
    if not float(spec["low"]) <= float(reference_value) <= float(spec["high"]):
        raise RuntimeError(
            f"Reference value {parameter_name}={reference_value} is outside "
            f"[{spec['low']}, {spec['high']}]."
        )


# ============================================================
# H. Fixed non-tuned settings
# ============================================================
FIXED_TRAINING_ARGS = {
    "optimizer": "SGD",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "nbs": 64,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.70,
    "hsv_v": 0.40,
    "degrees": 0.0,
    "translate": 0.10,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mixup": 0.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cache": CACHE_MODE,
    "cos_lr": False,
    "rect": False,
}


# ============================================================
# I. DEHB fidelity, evolutionary, and budget protocol
# ============================================================
DEHB_MIN_EPOCHS = SCREEN_EPOCHS
DEHB_MAX_EPOCHS = FULL_EPOCHS
DEHB_ETA = 2

# Method-specific optimizer seed, fixed across both cross-year directions.
DEHB_RANDOM_SEED = 789

# Two complete iterations match the 320 epoch-unit HPO budget used by RS and HB.
# Iteration 1 initializes the 20/40-epoch populations.
# Iteration 2 performs DE-guided evolution.
DEHB_ITERATIONS = 2

# Differential Evolution controls.
DE_MUTATION_FACTOR = 0.5
DE_CROSSOVER_RATE = 0.5

# Per iteration:
#   Bracket 1: 2 x 20 + 1 x 40 = 80 epoch-units
#   Bracket 2: 2 x 40          = 80 epoch-units
#   Total                       = 160 epoch-units
DEHB_BRACKET1_UNITS_PER_ITERATION = 2 * SCREEN_EPOCHS + 1 * FULL_EPOCHS
DEHB_BRACKET2_UNITS_PER_ITERATION = 2 * FULL_EPOCHS
DEHB_UNITS_PER_ITERATION = (
    DEHB_BRACKET1_UNITS_PER_ITERATION
    + DEHB_BRACKET2_UNITS_PER_ITERATION
)
DEHB_HPO_EPOCH_UNITS = DEHB_ITERATIONS * DEHB_UNITS_PER_ITERATION

EXPECTED_HPO_EVALUATIONS = 5 * DEHB_ITERATIONS
EXPECTED_FULL_FIDELITY_EVALUATIONS = 3 * DEHB_ITERATIONS
FINAL_EVALUATION_EPOCH_UNITS = len(FINAL_EVALUATION_SEEDS) * FINAL_EPOCHS

if DEHB_BRACKET1_UNITS_PER_ITERATION != 80:
    raise RuntimeError("DEHB bracket 1 must consume 80 epoch units per iteration.")
if DEHB_BRACKET2_UNITS_PER_ITERATION != 80:
    raise RuntimeError("DEHB bracket 2 must consume 80 epoch units per iteration.")
if DEHB_UNITS_PER_ITERATION != 160:
    raise RuntimeError("Each DEHB iteration must consume 160 epoch units.")
if DEHB_HPO_EPOCH_UNITS != 320:
    raise RuntimeError("Two DEHB iterations must consume exactly 320 HPO epoch units.")
if EXPECTED_HPO_EVALUATIONS != 10:
    raise RuntimeError("DEHB must perform exactly 10 HPO evaluations.")
if EXPECTED_FULL_FIDELITY_EVALUATIONS != 6:
    raise RuntimeError("DEHB must perform exactly 6 full fidelity evaluations.")
if FINAL_EVALUATION_EPOCH_UNITS != 120:
    raise RuntimeError("Final evaluation budget must be exactly 120 epoch units.")

REQUIRE_FULL_HPO_BUDGET = True
REQUIRE_ALL_FINAL_RUNS = True
STRICT_FITNESS_ALIGNMENT_CHECK = True
FITNESS_ALIGNMENT_TOLERANCE = 0.002


# ============================================================
# J. Run output folders
# ============================================================
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUNS_ROOT / "dehb20to40_2iterations" / RUN_TIMESTAMP
HPO_TRIALS_ROOT = RUN_ROOT / "hpo"
FINAL_TRAINING_ROOT = RUN_ROOT / "final40"
TARGET_TEST_EVAL_ROOT = RUN_ROOT / "target_test"
RUN_STATS_ROOT = RUN_ROOT / "stats"
RUN_PLOTS_ROOT = RUN_ROOT / "plots"



def privacy_safe_text(
    value: Any,
    workspace_root: Path,
) -> str:
    """Redact local workspace and user home paths from persisted text."""
    text = str(value)
    try:
        text = text.replace(str(workspace_root.resolve()), "<WORKSPACE>")
    except OSError:
        pass
    text = re.sub(r"[A-Za-z]:\\Users\\[^\\\s]+", "<USER_HOME>", text)
    text = re.sub(r"/home/[^/\s]+", "<USER_HOME>", text)
    text = re.sub(r"/Users/[^/\s]+", "<USER_HOME>", text)
    return text


def relative_output_path(
    path: Path,
    workspace_root: Path,
) -> str:
    """Return a portable path for saved manifests and result tables."""
    path = Path(path)
    try:
        return path.resolve().relative_to(workspace_root.resolve()).as_posix()
    except (ValueError, OSError):
        return path.name


print("=" * 92)
print("YOLOv8m DEHB: CROSS YEAR")
print("=" * 92)
print("Direction:", CROSS_YEAR_TAG)
print("Source dataset:", f"Datasets/{TRAIN_DATASET_TAG}")
print("Target dataset:", f"Datasets/{TEST_DATASET_TAG}")
print("Model:", MODEL_NAME)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Workers:", WORKERS)
print("Cache:", CACHE_MODE)
print("Patience:", PATIENCE)
print("HPO fitness: source validation mAP50-95 only")
print("DEHB fidelities:", DEHB_MIN_EPOCHS, "and", DEHB_MAX_EPOCHS)
print("DEHB iterations:", DEHB_ITERATIONS)
print("DEHB optimizer seed:", DEHB_RANDOM_SEED)
print("HPO budget:", DEHB_HPO_EPOCH_UNITS, "epoch units")
print("Expected HPO evaluations:", EXPECTED_HPO_EVALUATIONS)
print("Expected full fidelity evaluations:", EXPECTED_FULL_FIDELITY_EVALUATIONS)
print("Final evaluation budget:", FINAL_EVALUATION_EPOCH_UNITS, "epoch units, reported separately")
print("Optimization split seed:", OPTIMIZATION_SPLIT_SEED)
print("Optimization training seed:", OPTIMIZATION_TRAIN_SEED)
print("Final seeds:", FINAL_EVALUATION_SEEDS)
print("Python:", sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Target year test used during HPO: NO")
print("Output root:", relative_output_path(RUN_ROOT, WORKSPACE_ROOT))
print("=" * 92)


## Module 2: Annotation parsing and normalization

Provides the image, Pascal VOC XML, JSON, VIA, label normalization, image size, and bounding box helpers used to convert the original annotations into one clean internal record format.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def normalize_label(raw_label: str) -> str:
    """Normalize one class label using the fixed alias table."""
    cleaned = str(raw_label).strip()
    return LABEL_ALIASES.get(cleaned.lower(), cleaned)


def read_image_size(image_path: Path) -> Tuple[int, int]:
    """Return image width and height, failing clearly for unreadable files."""
    try:
        with Image.open(image_path) as image:
            return image.size
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError(f"Unreadable image: {image_path}\n{error}") from error


def parse_pascal_voc_xml(
    xml_path: Path,
) -> Tuple[Optional[Tuple[int, int]], List[Dict[str, Any]]]:
    """Parse one Pascal VOC XML annotation."""
    root = ET.parse(xml_path).getroot()
    xml_size = None
    size_node = root.find("size")

    if size_node is not None:
        width_text = size_node.findtext("width")
        height_text = size_node.findtext("height")
        if width_text and height_text:
            xml_size = (int(float(width_text)), int(float(height_text)))

    objects: List[Dict[str, Any]] = []
    for object_node in root.findall("object"):
        raw_name = object_node.findtext("name")
        box_node = object_node.find("bndbox")
        if not raw_name or box_node is None:
            continue

        coordinates = [
            box_node.findtext("xmin"),
            box_node.findtext("ymin"),
            box_node.findtext("xmax"),
            box_node.findtext("ymax"),
        ]
        if any(value is None for value in coordinates):
            continue

        try:
            x1, y1, x2, y2 = [float(value) for value in coordinates]
        except (TypeError, ValueError):
            continue

        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue

        objects.append({
            "label": normalize_label(raw_name),
            "bbox_xyxy": [x1, y1, x2, y2],
        })

    return xml_size, objects


def extract_via_label(attributes: Dict[str, Any]) -> Optional[str]:
    """Extract a label from common VIA region-attribute layouts."""
    preferred_keys = [
        "label", "class", "class_name", "name", "species", "type", "weed", "category"
    ]

    for key in preferred_keys:
        value = attributes.get(key)
        if isinstance(value, str) and value.strip():
            return normalize_label(value)
        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key)
            if len(value) == 1:
                return normalize_label(next(iter(value.keys())))

    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalize_label(value)

    return None


def parse_json_annotation(json_path: Path) -> List[Dict[str, Any]]:
    """Parse LabelMe-style or VIA-style JSON annotations."""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    objects: List[Dict[str, Any]] = []

    # LabelMe
    if isinstance(data, dict) and isinstance(data.get("shapes"), list):
        for shape in data["shapes"]:
            label = normalize_label(shape.get("label", ""))
            points = shape.get("points", [])
            if not label or len(points) < 2:
                continue

            xs = [float(point[0]) for point in points]
            ys = [float(point[1]) for point in points]
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            if x2 > x1 + 1 and y2 > y1 + 1:
                objects.append({"label": label, "bbox_xyxy": [x1, y1, x2, y2]})
        return objects

    # VIA
    metadata = data.get("_via_img_metadata", data) if isinstance(data, dict) else {}
    if not isinstance(metadata, dict):
        return objects

    for entry in metadata.values():
        if not isinstance(entry, dict):
            continue

        regions = entry.get("regions", [])
        if isinstance(regions, dict):
            regions = [regions] if "shape_attributes" in regions else list(regions.values())
        if not isinstance(regions, list):
            continue

        for region in regions:
            shape = region.get("shape_attributes", {}) or {}
            attributes = region.get("region_attributes", {}) or {}
            if shape.get("name") != "rect":
                continue

            values = [shape.get("x"), shape.get("y"), shape.get("width"), shape.get("height")]
            if any(value is None for value in values):
                continue

            label = extract_via_label(attributes)
            if not label:
                continue

            try:
                x, y, width, height = [float(value) for value in values]
            except (TypeError, ValueError):
                continue

            if width > 1 and height > 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x, y, x + width, y + height],
                })

    return objects


def clip_box(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Optional[List[float]]:
    """Clip one xyxy box to image boundaries."""
    x1, y1, x2, y2 = box
    x1 = max(0.0, min(float(x1), image_width - 1.0))
    y1 = max(0.0, min(float(y1), image_height - 1.0))
    x2 = max(0.0, min(float(x2), float(image_width)))
    y2 = max(0.0, min(float(y2), float(image_height)))

    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None
    return [x1, y1, x2, y2]

## Module 3: Build and audit clean source and target records

Scans each selected year, keeps only the eight target classes, removes invalid boxes, computes dataset audits, and provides one source versus target class distribution plot. Dataset scanning happens only when the complete experiment is launched.


In [ ]:
def build_clean_records(
    source_root: Path,
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Build clean detection records from one source dataset folder."""
    source_root = Path(source_root)
    if not source_root.exists():
        raise FileNotFoundError(f"Source dataset not found: {source_root.name}")

    image_root = next(
        (p for p in [source_root / "JPEGImages", source_root / "images", source_root] if p.exists()),
        None,
    )
    if image_root is None:
        raise FileNotFoundError(f"No image root found below: {source_root.name}")

    image_paths = sorted(
        p for p in image_root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No images found below: {source_root.name}")

    image_names = [p.name for p in image_paths]
    if len(image_names) != len(set(image_names)):
        raise ValueError(
            f"Duplicate image filenames found below {source_root.name}; "
            "flat YOLO split folders require unique filenames within each dataset."
        )

    xml_by_stem = {p.stem: p for p in source_root.rglob("*.xml")}
    json_by_stem = {p.stem: p for p in source_root.rglob("*.json")}

    records: List[Dict[str, Any]] = []
    stats = {
        "images_found": len(image_paths),
        "records_kept": 0,
        "missing_annotation": 0,
        "empty_after_filtering": 0,
        "unreadable_images": 0,
        "invalid_boxes_removed": 0,
        "non_target_objects_removed": 0,
        "used_xml": 0,
        "used_json": 0,
    }

    for image_path in image_paths:
        xml_path = xml_by_stem.get(image_path.stem)
        json_path = json_by_stem.get(image_path.stem)
        annotation_size = None
        objects: List[Dict[str, Any]] = []
        source_used = None

        try:
            if xml_path is not None:
                annotation_size, xml_objects = parse_pascal_voc_xml(xml_path)
                if xml_objects:
                    objects = xml_objects
                    source_used = "xml"

            # Fallback to JSON when XML is absent or produced no usable objects.
            if not objects and json_path is not None:
                objects = parse_json_annotation(json_path)
                if objects:
                    source_used = "json"
                    annotation_size = None

            if not objects:
                stats["missing_annotation"] += 1
                continue

            if annotation_size is None:
                width, height = read_image_size(image_path)
            else:
                width, height = annotation_size

        except Exception as error:
            stats["unreadable_images"] += 1
            print(f"Skipping {image_path.name}: {error}")
            continue

        clean_objects: List[Dict[str, Any]] = []
        for obj in objects:
            label = normalize_label(obj["label"])
            if label not in TARGET_CLASS_SET:
                stats["non_target_objects_removed"] += 1
                continue

            clipped = clip_box(obj["bbox_xyxy"], width, height)
            if clipped is None:
                stats["invalid_boxes_removed"] += 1
                continue

            clean_objects.append({"label": label, "bbox_xyxy": clipped})

        if not clean_objects:
            stats["empty_after_filtering"] += 1
            continue

        stats[f"used_{source_used}"] += 1
        records.append({
            "img_path": str(image_path.resolve()),
            "file_name": image_path.name,
            "width": int(width),
            "height": int(height),
            "objects": clean_objects,
        })

    records.sort(key=lambda row: row["file_name"])
    stats["records_kept"] = len(records)

    if not records:
        raise RuntimeError(f"No usable records remained after filtering: {source_root}")

    return records, stats


def audit_records(
    records_to_audit: List[Dict[str, Any]],
    class_names: List[str],
) -> Dict[str, Any]:
    """Count images, boxes, and class occurrence statistics."""
    box_counts = {name: 0 for name in class_names}
    image_counts = {name: 0 for name in class_names}

    for record in records_to_audit:
        labels = [obj["label"] for obj in record["objects"]]
        for label in labels:
            box_counts[label] += 1
        for label in set(labels):
            image_counts[label] += 1

    return {
        "num_images": len(records_to_audit),
        "num_boxes": int(sum(box_counts.values())),
        "box_count_per_class": box_counts,
        "image_count_per_class": image_counts,
        "missing_classes": [name for name in class_names if box_counts[name] == 0],
    }


def plot_source_target_box_counts(
    source_audit: Dict[str, Any],
    target_audit: Dict[str, Any],
    output_root: Path,
) -> None:
    """Plot one grouped comparison of full-dataset box counts by class."""
    output_root.mkdir(parents=True, exist_ok=True)
    x = np.arange(len(CLASS_NAMES))
    width = 0.38

    plt.figure(figsize=(11, 5))
    plt.bar(
        x - width / 2,
        [source_audit["box_count_per_class"][name] for name in CLASS_NAMES],
        width,
        label=f"{TRAIN_DATASET_TAG} source",
    )
    plt.bar(
        x + width / 2,
        [target_audit["box_count_per_class"][name] for name in CLASS_NAMES],
        width,
        label=f"{TEST_DATASET_TAG} target",
    )
    plt.xticks(x, CLASS_NAMES, rotation=45, ha="right")
    plt.ylabel("Number of annotated boxes")
    plt.title(f"{CROSS_YEAR_TAG}: full-dataset class counts")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_root / "source_target_box_counts.png", dpi=180, bbox_inches="tight")
    plt.show()




## Module 4: Deterministic cross year splitting and source train subsampling

Creates independent 70/15/15 source and target splits. The effective experiment uses source train for training, source validation for HPO and checkpoint selection, and target test only for final cross year evaluation. When Year2021 is the source, only its training split is subsampled to the target training split size using the established stratified rule.


In [ ]:
def split_records_deterministically(
    all_records: List[Dict[str, Any]],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    seed: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Create one reproducible 70/15/15 split."""
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0.")

    shuffled = list(all_records)
    random.Random(seed).shuffle(shuffled)

    total = len(shuffled)
    n_train = int(total * train_ratio)
    n_valid = int(total * valid_ratio)

    return {
        "train": shuffled[:n_train],
        "valid": shuffled[n_train:n_train + n_valid],
        "test": shuffled[n_train + n_valid:],
    }


def get_record_primary_label(record: Dict[str, Any]) -> str:
    """
    Assign one primary class to an image for stratified subsampling.

    For multi-class images, use the most frequent class.
    Ties follow the fixed CLASS_NAMES order.
    """
    labels = [obj["label"] for obj in record.get("objects", [])]
    if not labels:
        return "unknown"

    counts = Counter(labels)
    return max(
        CLASS_NAMES,
        key=lambda class_name: (
            counts.get(class_name, 0),
            -CLASS_NAMES.index(class_name),
        ),
    )


def _exact_stratified_allocation(
    groups: Dict[str, List[Dict[str, Any]]],
    target_size: int,
) -> Dict[str, int]:
    """
    Allocate an exact target size approximately in proportion to group sizes,
    while retaining at least one record per represented group when possible.
    """
    labels = [label for label, group in groups.items() if group]
    total = sum(len(groups[label]) for label in labels)

    if target_size < 0 or target_size > total:
        raise ValueError("target_size must be between 0 and the number of available records.")
    if target_size == 0:
        return {label: 0 for label in labels}

    raw = {label: target_size * len(groups[label]) / total for label in labels}
    allocation = {label: min(int(math.floor(raw[label])), len(groups[label])) for label in labels}

    # Preserve represented classes where the requested sample is large enough.
    if target_size >= len(labels):
        for label in labels:
            if allocation[label] == 0:
                allocation[label] = 1

    # If minimum-one adjustments overshoot, remove from the most over-allocated
    # groups while keeping at least one when that rule is active.
    min_allowed = 1 if target_size >= len(labels) else 0
    while sum(allocation.values()) > target_size:
        reducible = [
            label for label in labels
            if allocation[label] > min_allowed
        ]
        if not reducible:
            raise RuntimeError("Could not reduce stratified allocation to the exact target size.")
        label = max(
            reducible,
            key=lambda name: (allocation[name] - raw[name], allocation[name], len(groups[name])),
        )
        allocation[label] -= 1

    # Fill remaining slots using largest remainder, respecting group capacity.
    while sum(allocation.values()) < target_size:
        candidates = [
            label for label in labels
            if allocation[label] < len(groups[label])
        ]
        if not candidates:
            raise RuntimeError("Could not fill stratified allocation to the exact target size.")
        label = max(
            candidates,
            key=lambda name: (raw[name] - allocation[name], len(groups[name])),
        )
        allocation[label] += 1

    if sum(allocation.values()) != target_size:
        raise RuntimeError("Stratified allocation did not reach the requested target size.")

    return allocation


def stratified_subsample_records(
    records: List[Dict[str, Any]],
    target_size: int,
    seed: int,
) -> List[Dict[str, Any]]:
    """Stratified image-level subsampling using the primary class of each image."""
    if target_size >= len(records):
        return list(records)

    rng = random.Random(seed)
    groups: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    for record in records:
        groups[get_record_primary_label(record)].append(record)

    for group in groups.values():
        rng.shuffle(group)

    allocation = _exact_stratified_allocation(groups, target_size)

    sampled: List[Dict[str, Any]] = []
    for label, group in groups.items():
        sampled.extend(group[:allocation.get(label, 0)])

    rng.shuffle(sampled)

    if len(sampled) != target_size:
        raise RuntimeError(
            f"Subsampling produced {len(sampled)} records; expected exactly {target_size}."
        )
    return sampled


def maybe_subsample_source_train(
    source_train_records: List[Dict[str, Any]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    seed: int,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Apply the established subsampling rule only when Year2021 is the source year."""
    should_subsample = (
        SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE
        and TRAIN_YEAR == "2021"
    )

    info = {
        "enabled": bool(should_subsample),
        "source_year": TRAIN_YEAR,
        "target_year": TEST_YEAR,
        "mode": SUBSAMPLE_TARGET_MODE,
        "subsample_seed": None,
        "original_source_train_count": len(source_train_records),
        "target_size": None,
        "final_source_train_count": len(source_train_records),
    }

    if not should_subsample:
        return list(source_train_records), info

    if SUBSAMPLE_TARGET_MODE != "target_train_size":
        raise ValueError("This notebook freezes SUBSAMPLE_TARGET_MODE='target_train_size'.")

    target_size = len(target_splits["train"])
    subsample_seed = int(seed + SUBSAMPLE_SEED_OFFSET)

    if target_size > len(source_train_records):
        # Do not upsample. This guard makes the rule explicit and auditable.
        print(
            "WARNING: target train size exceeds Year2021 source train size; "
            "subsampling is skipped because the protocol does not upsample."
        )
        info["target_size"] = int(target_size)
        info["subsample_seed"] = subsample_seed
        return list(source_train_records), info

    sampled = stratified_subsample_records(
        records=source_train_records,
        target_size=target_size,
        seed=subsample_seed,
    )

    info["target_size"] = int(target_size)
    info["subsample_seed"] = subsample_seed
    info["final_source_train_count"] = len(sampled)
    return sampled, info


def _path_set(records: List[Dict[str, Any]]) -> set:
    return {str(Path(record["img_path"]).resolve()) for record in records}


def verify_cross_split_disjointness(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Fail if any physical image path occurs in more than one effective split."""
    train_paths = _path_set(cross_year_splits["train"])
    valid_paths = _path_set(cross_year_splits["valid"])
    test_paths = _path_set(cross_year_splits["test"])

    overlaps = {
        "train_valid": train_paths & valid_paths,
        "train_test": train_paths & test_paths,
        "valid_test": valid_paths & test_paths,
    }
    bad = {name: paths for name, paths in overlaps.items() if paths}
    if bad:
        raise RuntimeError(
            "Cross-year split leakage detected: "
            + ", ".join(f"{name}={len(paths)}" for name, paths in bad.items())
        )


def audit_cross_year_splits(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    fail_on_missing: bool = True,
) -> Dict[str, Any]:
    """Audit effective train/valid/test class coverage."""
    report: Dict[str, Any] = {}
    for split_name, rows in cross_year_splits.items():
        split_audit = audit_records(rows, CLASS_NAMES)
        report[split_name] = split_audit
        print(
            f"{split_name:8s} images={split_audit['num_images']:5d} "
            f"boxes={split_audit['num_boxes']:6d} "
            f"missing={split_audit['missing_classes']}"
        )
        if fail_on_missing and split_audit["missing_classes"]:
            raise RuntimeError(
                f"Effective split '{split_name}' is missing classes: "
                f"{split_audit['missing_classes']}"
            )
    return report


def make_cross_year_splits_for_seed(
    seed: int,
    source_records: List[Dict[str, Any]],
    target_records: List[Dict[str, Any]],
) -> Tuple[
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, Any],
]:
    """
    Build source splits, target splits, and the effective cross-year layout.

    Effective YOLO layout:
        train = source train (subsampled only if source is Year2021)
        valid = source validation
        test  = target test
    """
    source_splits = split_records_deterministically(
        source_records, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )
    target_splits = split_records_deterministically(
        target_records, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )

    effective_source_train, subsample_info = maybe_subsample_source_train(
        source_train_records=source_splits["train"],
        target_splits=target_splits,
        seed=seed,
    )

    cross_year_splits = {
        "train": effective_source_train,
        "valid": source_splits["valid"],
        "test": target_splits["test"],
    }

    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)

    split_info = {
        "seed": int(seed),
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "source_dataset": TRAIN_DATASET_TAG,
        "target_dataset": TEST_DATASET_TAG,
        "ratios": {
            "train": TRAIN_RATIO,
            "valid": VALID_RATIO,
            "test": TEST_RATIO,
        },
        "source_counts_original": {
            name: len(rows) for name, rows in source_splits.items()
        },
        "target_counts_original": {
            name: len(rows) for name, rows in target_splits.items()
        },
        "effective_cross_year_counts": {
            name: len(rows) for name, rows in cross_year_splits.items()
        },
        "subsample_info": subsample_info,
        "class_coverage": coverage,
    }

    return cross_year_splits, source_splits, target_splits, split_info




## Module 5: YOLO dataset export and integrity checks

Exports the effective train, validation, and test layout in YOLO format, writes filename only split manifests, verifies image and label counts, and checks the effective splits again before training begins.


In [ ]:
def remove_file_or_link(path: Path) -> None:
    """Remove a regular file or symlink, including broken symlinks."""
    if path.is_symlink() or path.is_file():
        path.unlink()


def clear_split_directory(directory: Path) -> None:
    """Clear files, symlinks, and old disk-cache .npy files from one split folder."""
    directory.mkdir(parents=True, exist_ok=True)
    for path in directory.iterdir():
        if path.is_file() or path.is_symlink():
            path.unlink()


def place_image(source: Path, destination: Path, mode: str) -> str:
    """Place one image using symlink -> hardlink -> copy fallback."""
    if mode not in {"symlink", "hardlink", "copy"}:
        raise ValueError("mode must be 'symlink', 'hardlink', or 'copy'.")

    source = source.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    remove_file_or_link(destination)

    if mode == "symlink":
        try:
            destination.symlink_to(source)
            return "symlinked"
        except OSError:
            pass

    if mode in {"symlink", "hardlink"}:
        try:
            os.link(source, destination)
            return "hardlinked"
        except OSError:
            pass

    shutil.copy2(source, destination)
    return "copied"


def xyxy_to_yolo(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Tuple[float, float, float, float]:
    """Convert absolute xyxy coordinates to normalized YOLO xywh."""
    x1, y1, x2, y2 = box
    values = (
        ((x1 + x2) / 2.0) / image_width,
        ((y1 + y2) / 2.0) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )
    return tuple(max(0.0, min(1.0, float(value))) for value in values)


def write_yolo_label(
    record: Dict[str, Any],
    output_path: Path,
    class_to_id: Dict[str, int],
) -> None:
    """Write one YOLO label file."""
    lines: List[str] = []
    for obj in record["objects"]:
        class_id = class_to_id[obj["label"]]
        xc, yc, bw, bh = xyxy_to_yolo(
            obj["bbox_xyxy"], record["width"], record["height"]
        )
        lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_dataset_yaml(
    dataset_root: Path,
    class_names: List[str],
) -> Path:
    """Write an Ultralytics dataset.yaml for the effective cross-year layout."""
    yaml_path = dataset_root / "dataset.yaml"
    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/valid",
        "test: images/test",
        f"nc: {len(class_names)}",
        "names:",
    ]
    lines.extend(f"  {idx}: {name}" for idx, name in enumerate(class_names))
    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return yaml_path


def save_split_manifest(
    split_records: Dict[str, List[Dict[str, Any]]],
    output_json: Path,
    seed: int,
    dataset_tag: str,
    role: str,
    extra_info: Optional[Dict[str, Any]] = None,
) -> None:
    """Save filenames and counts for reproducibility."""
    output_json.parent.mkdir(parents=True, exist_ok=True)
    manifest = {
        "dataset_tag": dataset_tag,
        "role": role,
        "seed": int(seed),
        "counts": {name: len(rows) for name, rows in split_records.items()},
        "files": {
            name: [record["file_name"] for record in rows]
            for name, rows in split_records.items()
        },
    }
    if extra_info is not None:
        manifest["extra_info"] = extra_info
    output_json.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def count_image_files(directory: Path) -> int:
    """Count only actual image files/symlinks, excluding disk-cache .npy files."""
    return sum(
        1 for path in directory.iterdir()
        if (path.is_file() or path.is_symlink())
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def count_label_files(directory: Path) -> int:
    return sum(
        1 for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".txt"
    )


def check_yolo_dataset_integrity(
    dataset_root: Path,
    expected_counts: Dict[str, int],
) -> None:
    """Verify expected image and label counts for every effective split."""
    for split_name in ("train", "valid", "test"):
        image_dir = dataset_root / "images" / split_name
        label_dir = dataset_root / "labels" / split_name
        image_count = count_image_files(image_dir)
        label_count = count_label_files(label_dir)
        expected = int(expected_counts[split_name])

        print(
            f"{split_name:8s} images={image_count:5d} "
            f"labels={label_count:5d} expected={expected:5d}"
        )
        if image_count != expected or label_count != expected:
            raise RuntimeError(
                f"YOLO integrity failure in '{split_name}': "
                f"images={image_count}, labels={label_count}, expected={expected}."
            )


def prepare_cross_year_yolo_dataset(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    source_splits: Dict[str, List[Dict[str, Any]]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    split_info: Dict[str, Any],
    output_root: Path,
    manifest_root: Path,
    image_mode: str,
    seed: int,
) -> Tuple[Path, Dict[str, Any]]:
    """Build and verify one effective cross-year YOLO dataset."""
    output_root = Path(output_root)
    manifest_root = Path(manifest_root)
    class_to_id = {name: idx for idx, name in enumerate(CLASS_NAMES)}

    # Recheck leakage and class coverage immediately before export.
    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)

    image_dirs = {
        name: output_root / "images" / name for name in ("train", "valid", "test")
    }
    label_dirs = {
        name: output_root / "labels" / name for name in ("train", "valid", "test")
    }

    for directory in [*image_dirs.values(), *label_dirs.values()]:
        clear_split_directory(directory)

    placement_counts = {"symlinked": 0, "hardlinked": 0, "copied": 0}

    for split_name, rows in cross_year_splits.items():
        for record in rows:
            source = Path(record["img_path"])
            destination = image_dirs[split_name] / record["file_name"]
            action = place_image(source, destination, image_mode)
            placement_counts[action] += 1

            label_path = label_dirs[split_name] / f"{Path(record['file_name']).stem}.txt"
            write_yolo_label(record, label_path, class_to_id)

    yaml_path = write_dataset_yaml(output_root, CLASS_NAMES)

    save_split_manifest(
        source_splits,
        manifest_root / f"{TRAIN_DATASET_TAG}_source_seed{seed}.json",
        seed,
        TRAIN_DATASET_TAG,
        "source",
    )
    save_split_manifest(
        target_splits,
        manifest_root / f"{TEST_DATASET_TAG}_target_seed{seed}.json",
        seed,
        TEST_DATASET_TAG,
        "target",
    )
    save_split_manifest(
        cross_year_splits,
        manifest_root / f"cross_{CROSS_YEAR_TAG}_seed{seed}.json",
        seed,
        CROSS_YEAR_TAG,
        "effective_cross_year",
        extra_info=split_info,
    )

    expected_counts = {
        name: len(rows) for name, rows in cross_year_splits.items()
    }
    check_yolo_dataset_integrity(output_root, expected_counts)

    layout_manifest = {
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "seed": int(seed),
        "dataset_yaml": relative_output_path(yaml_path, WORKSPACE_ROOT),
        "output_root": relative_output_path(output_root, WORKSPACE_ROOT),
        "placement_counts": placement_counts,
        "expected_counts": expected_counts,
        "split_info": split_info,
    }
    manifest_root.mkdir(parents=True, exist_ok=True)
    (manifest_root / "yolo_layout_manifest.json").write_text(
        json.dumps(layout_manifest, indent=2), encoding="utf-8"
    )

    print("Dataset YAML:", relative_output_path(yaml_path, WORKSPACE_ROOT))
    print("Image placement:", placement_counts)
    return yaml_path, coverage

## Module 6: Validation metrics and YOLO training helper

Defines metric extraction, checkpoint alignment checks, and the shared YOLOv8m training helper. HPO fitness is source validation mAP50-95 only. Every DEHB evaluation starts from the pretrained yolov8m.pt weights.


In [ ]:
def extract_box_metrics(
    metrics: Any,
    expected_class_names: List[str],
) -> Dict[str, Any]:
    """Extract detection metrics; optimization fitness is mAP50-95 only."""
    box = getattr(metrics, "box", None)
    if box is None:
        raise RuntimeError("Ultralytics returned no detection box metrics.")

    precision = float(box.mp)
    recall = float(box.mr)
    map50 = float(box.map50)
    map50_95 = float(box.map)

    names_obj = getattr(metrics, "names", {})
    if isinstance(names_obj, dict):
        class_name_lookup = {int(k): str(v) for k, v in names_obj.items()}
    else:
        class_name_lookup = {idx: str(name) for idx, name in enumerate(names_obj)}

    per_class_rows: List[Dict[str, Any]] = []

    if hasattr(box, "ap") and hasattr(box, "ap_class_index"):
        ap_values = np.asarray(box.ap, dtype=float).reshape(-1)
        class_ids = np.asarray(box.ap_class_index, dtype=int).reshape(-1)

        for class_id, ap_value in zip(class_ids.tolist(), ap_values.tolist()):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })
    else:
        maps = [float(value) for value in box.maps]
        for class_id, ap_value in enumerate(maps):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })

    represented_ids = {int(row["class_id"]) for row in per_class_rows}
    expected_ids = set(range(len(expected_class_names)))
    missing_ids = sorted(expected_ids - represented_ids)

    if missing_ids:
        missing_names = [expected_class_names[idx] for idx in missing_ids]
        raise RuntimeError(
            "Validation/test metrics are missing represented classes: "
            f"{missing_names}. Check the split coverage."
        )

    sorted_aps = sorted(float(row["map50_95"]) for row in per_class_rows)
    bottom3 = float(mean(sorted_aps[:3])) if len(sorted_aps) >= 3 else float(mean(sorted_aps))

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "fitness": map50_95,
        "bottom3_mean_map50_95": bottom3,
        "per_class_map50_95": per_class_rows,
    }


def read_epoch_map50_95_values(results_csv: Path) -> Tuple[str, List[float]]:
    """Read epoch-level mAP50-95 values from an Ultralytics results.csv file."""
    results_csv = Path(results_csv)
    if not results_csv.exists():
        raise FileNotFoundError(f"Training results.csv not found: {results_csv}")

    with results_csv.open("r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    candidate_columns = [name for name in fieldnames if "mAP50-95" in name]
    if not candidate_columns:
        raise RuntimeError(
            f"Could not find an mAP50-95 column in {results_csv}. "
            f"Available columns: {fieldnames}"
        )

    metric_column = next(
        (name for name in candidate_columns if "metrics/" in name),
        candidate_columns[0],
    )

    values: List[float] = []
    for row in rows:
        raw_value = row.get(metric_column, "")
        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(value):
            values.append(value)

    if not values:
        raise RuntimeError(f"No numeric values found in results column: {metric_column}")

    return metric_column, values


def verify_ultralytics_fitness_alignment(
    results_csv: Path,
    trainer_best_fitness: Optional[float],
    tolerance: float,
    strict: bool,
) -> Dict[str, Any]:
    """Verify that trainer.best_fitness agrees with maximum epoch mAP50-95."""
    metric_column, map95_values = read_epoch_map50_95_values(results_csv)
    max_epoch_map95 = float(max(map95_values))

    if trainer_best_fitness is None or not math.isfinite(float(trainer_best_fitness)):
        report = {
            "status": "UNVERIFIED",
            "metric_column": metric_column,
            "max_epoch_map50_95": max_epoch_map95,
            "trainer_best_fitness": None,
            "absolute_difference": None,
            "tolerance": float(tolerance),
        }
        if strict:
            raise RuntimeError(
                "Could not read Ultralytics trainer.best_fitness, so checkpoint-fitness "
                "alignment could not be verified."
            )
        return report

    trainer_best_fitness = float(trainer_best_fitness)
    difference = abs(trainer_best_fitness - max_epoch_map95)
    status = "PASS" if difference <= tolerance else "FAIL"

    report = {
        "status": status,
        "metric_column": metric_column,
        "max_epoch_map50_95": max_epoch_map95,
        "trainer_best_fitness": trainer_best_fitness,
        "absolute_difference": float(difference),
        "tolerance": float(tolerance),
    }

    if strict and status != "PASS":
        raise RuntimeError(
            "Ultralytics checkpoint fitness does not align with mAP50-95. "
            f"best_fitness={trainer_best_fitness:.6f}, "
            f"max epoch mAP50-95={max_epoch_map95:.6f}, "
            f"difference={difference:.6f}."
        )

    return report


def train_and_validate_configuration(
    *,
    learning_rate: float,
    lrf: float,
    momentum: float,
    weight_decay: float,
    scale: float,
    dataset_yaml: Path,
    project_dir: Path,
    model_name: str,
    epochs: int,
    image_size: int,
    batch_size: int,
    patience: int,
    workers: int,
    seed: int,
    warmup_epochs: float,
    close_mosaic: int,
    fixed_training_args: Dict[str, Any],
    expected_class_names: List[str],
    strict_fitness_alignment_check: bool,
    fitness_alignment_tolerance: float,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Train from scratch, verify checkpoint alignment, then validate best.pt."""
    dataset_yaml = Path(dataset_yaml)
    project_dir = Path(project_dir)

    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")
    if learning_rate <= 0 or lrf <= 0:
        raise ValueError("learning_rate and lrf must be greater than 0.")
    if not 0.0 < momentum < 1.0:
        raise ValueError("momentum must be between 0 and 1.")
    if weight_decay < 0:
        raise ValueError("weight_decay cannot be negative.")
    if not 0.0 <= scale <= 1.0:
        raise ValueError("scale must be between 0 and 1.")
    if epochs < 1 or batch_size < 1 or workers < 0:
        raise ValueError("Invalid epochs, batch_size, or workers value.")
    if warmup_epochs < 0 or close_mosaic < 0:
        raise ValueError("warmup_epochs and close_mosaic must be non-negative.")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"
    using_cuda = str(device) != "cpu" and torch.cuda.is_available()

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    trial_id = uuid.uuid4().hex[:8]
    run_name = f"CY{TRAIN_YEAR}{TEST_YEAR}_s{seed}_e{epochs}_{trial_id}"
    project_dir.mkdir(parents=True, exist_ok=True)

    hyperparameters = {
        "learning_rate": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "epochs": int(epochs),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
        "seed": int(seed),
        "patience": int(patience),
    }

    print("\n" + "=" * 88)
    print("START TRAINING:", run_name)
    print(json.dumps(hyperparameters, indent=2))
    print("=" * 88)

    train_kwargs = dict(fixed_training_args)
    train_kwargs.update({
        "data": str(dataset_yaml.resolve()),
        "project": str(project_dir.resolve()),
        "name": run_name,
        "epochs": int(epochs),
        "imgsz": int(image_size),
        "batch": int(batch_size),
        "patience": int(patience),
        "workers": int(workers),
        "device": device,
        "seed": int(seed),
        "deterministic": True,
        "pretrained": True,
        "val": True,
        "save": True,
        "save_period": -1,
        "plots": False,
        "verbose": True,
        "amp": bool(using_cuda),
        "lr0": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
    })

    model = YOLO(model_name)
    train_start = time.perf_counter()
    train_result = model.train(**train_kwargs)
    training_seconds = time.perf_counter() - train_start

    train_save_dir = getattr(train_result, "save_dir", None)
    if train_save_dir is None and getattr(model, "trainer", None) is not None:
        train_save_dir = getattr(model.trainer, "save_dir", None)
    if train_save_dir is None:
        raise RuntimeError("Training finished but save_dir could not be resolved.")

    train_save_dir = Path(train_save_dir)
    best_checkpoint = train_save_dir / "weights" / "best.pt"
    results_csv = train_save_dir / "results.csv"

    if not best_checkpoint.exists():
        raise FileNotFoundError(f"best.pt not found: {best_checkpoint}")
    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv}")

    trainer_best_fitness = None
    trainer = getattr(model, "trainer", None)
    if trainer is not None:
        raw_best_fitness = getattr(trainer, "best_fitness", None)
        if raw_best_fitness is not None:
            trainer_best_fitness = float(raw_best_fitness)

    alignment_report = verify_ultralytics_fitness_alignment(
        results_csv=results_csv,
        trainer_best_fitness=trainer_best_fitness,
        tolerance=fitness_alignment_tolerance,
        strict=strict_fitness_alignment_check,
    )

    # Re-evaluate the frozen best checkpoint on the SOURCE validation split.
    best_model = YOLO(str(best_checkpoint))
    val_start = time.perf_counter()
    validation_metrics = best_model.val(
        data=str(dataset_yaml.resolve()),
        split="val",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(project_dir.resolve()),
        name=f"{run_name}_val",
        plots=False,
        save_json=False,
        verbose=False,
    )
    validation_seconds = time.perf_counter() - val_start

    metric_data = extract_box_metrics(validation_metrics, expected_class_names)

    peak_gpu_memory_gb = None
    if using_cuda:
        peak_gpu_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    result = {
        "status": "PASS",
        "trial_id": trial_id,
        "run_name": run_name,
        "cross_year_tag": CROSS_YEAR_TAG,
        "hyperparameters": hyperparameters,
        "metrics": metric_data,
        "fitness_alignment": alignment_report,
        "timing": {
            "training_seconds": float(training_seconds),
            "validation_seconds": float(validation_seconds),
            "total_seconds": float(training_seconds + validation_seconds),
        },
        "hardware": {
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0) if using_cuda else None,
            "peak_gpu_memory_gb": (
                float(peak_gpu_memory_gb) if peak_gpu_memory_gb is not None else None
            ),
        },
        "paths": {
            "training_folder": train_save_dir,
            "best_checkpoint": best_checkpoint,
            "results_csv": results_csv,
        },
    }

    persisted_result = dict(result)
    persisted_result["paths"] = {
        key: relative_output_path(Path(value), WORKSPACE_ROOT)
        for key, value in result["paths"].items()
    }
    (train_save_dir / "trial_summary.json").write_text(
        json.dumps(persisted_result, indent=2), encoding="utf-8"
    )

    print(
        f"Completed {run_name}: source-val mAP50-95={metric_data['map50_95']:.6f}, "
        f"P={metric_data['precision']:.6f}, R={metric_data['recall']:.6f}, "
        f"alignment={alignment_report['status']}"
    )
    return result

## Module 7: DEHB search mechanics and exact budget plan

Defines normalized parameter encoding, Differential Evolution mutation and crossover, fidelity populations, ranking, result persistence, and the fixed two iteration DEHB schedule. The HPO budget is audited to exactly 320 epoch units.


In [ ]:
def safe_float(value: Any) -> Optional[float]:
    """Convert to float or return None."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def save_rows_to_csv_safe(rows: List[Dict[str, Any]], output_csv: Path) -> None:
    """Save heterogeneous dictionaries to CSV with a safe Excel-open fallback."""
    if not rows:
        return
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: List[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)

    try:
        with output_csv.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    except PermissionError:
        backup = output_csv.with_name(
            f"{output_csv.stem}_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}{output_csv.suffix}"
        )
        with backup.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print("CSV was open elsewhere; backup written to:", backup)


def save_json(data: Dict[str, Any], output_json: Path) -> None:
    """Save JSON with parent-directory creation."""
    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(data, indent=2), encoding="utf-8")


def summarize_metric_mean_sd(
    rows: List[Dict[str, Any]],
    metric_names: List[str],
) -> List[Dict[str, Any]]:
    """Return mean/sample-SD/min/max summary rows."""
    summary: List[Dict[str, Any]] = []
    for metric_name in metric_names:
        values = [
            value for value in (safe_float(row.get(metric_name)) for row in rows)
            if value is not None
        ]
        if not values:
            summary.append({"metric": metric_name, "n": 0, "mean": "", "sd": "", "min": "", "max": ""})
        elif len(values) == 1:
            summary.append({"metric": metric_name, "n": 1, "mean": values[0], "sd": "", "min": values[0], "max": values[0]})
        else:
            summary.append({
                "metric": metric_name,
                "n": len(values),
                "mean": mean(values),
                "sd": stdev(values),
                "min": min(values),
                "max": max(values),
            })
    return summary


def summarize_per_class_mean_sd(
    per_class_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Summarize target-test per-class mAP50-95 across final seeds."""
    summary: List[Dict[str, Any]] = []

    for class_name in sorted({row["class_name"] for row in per_class_rows}):
        values = [
            float(row["target_test_map50_95"])
            for row in per_class_rows
            if row["class_name"] == class_name
        ]

        summary.append({
            "class_name": class_name,
            "n": len(values),
            "mean_target_test_map50_95": mean(values),
            "sd_target_test_map50_95": "" if len(values) == 1 else stdev(values),
            "min_target_test_map50_95": min(values),
            "max_target_test_map50_95": max(values),
        })

    return summary


def boundary_proximity_report(
    selected_config: Dict[str, Any],
    search_space: Dict[str, Dict[str, Any]],
    parameter_order: List[str],
    boundary_fraction: float = 0.10,
) -> List[Dict[str, Any]]:
    """Flag winner parameters lying in the lower/upper 10% of their ranges."""
    report: List[Dict[str, Any]] = []
    for name in parameter_order:
        low = float(search_space[name]["low"])
        high = float(search_space[name]["high"])
        value = float(selected_config[name])

        if search_space[name]["distribution"] == "log":
            low_t, high_t, value_t = math.log(low), math.log(high), math.log(value)
        else:
            low_t, high_t, value_t = low, high, value

        fraction = (value_t - low_t) / (high_t - low_t)
        if fraction <= boundary_fraction:
            boundary = "LOW"
        elif fraction >= 1.0 - boundary_fraction:
            boundary = "HIGH"
        else:
            boundary = "INTERIOR"

        report.append({
            "parameter": name,
            "value": value,
            "low": low,
            "high": high,
            "normalized_position": fraction,
            "boundary_flag": boundary,
        })
    return report


def get_fidelity_controls(budget_epochs: int) -> Tuple[float, int]:
    """Return the frozen warmup and close-mosaic settings for a DEHB fidelity."""
    budget_epochs = int(budget_epochs)
    if budget_epochs == SCREEN_EPOCHS:
        return float(SCREEN_WARMUP_EPOCHS), int(SCREEN_CLOSE_MOSAIC)
    if budget_epochs == FULL_EPOCHS:
        return float(FULL_WARMUP_EPOCHS), int(FULL_CLOSE_MOSAIC)
    raise ValueError(
        f"Unsupported DEHB fidelity: {budget_epochs}. "
        f"Only {SCREEN_EPOCHS} and {FULL_EPOCHS} epochs are allowed."
    )


def sample_parameter(
    rng: random.Random,
    spec: Dict[str, Any],
) -> float:
    """Sample one declared parameter using its linear or log geometry."""
    low = float(spec["low"])
    high = float(spec["high"])
    distribution = str(spec["distribution"]).lower()

    if distribution == "log":
        return float(math.exp(rng.uniform(math.log(low), math.log(high))))
    if distribution == "linear":
        return float(rng.uniform(low, high))
    raise ValueError(f"Unsupported distribution: {distribution}")


def sample_dehb_config(
    *,
    rng: random.Random,
    config_id: str,
) -> Dict[str, Any]:
    """Sample one initial focused-5D configuration."""
    config = {"config_id": str(config_id)}
    for name in SEARCH_PARAMETER_ORDER:
        config[name] = sample_parameter(rng, SEARCH_SPACE[name])
    return config


def config_signature(config: Dict[str, Any]) -> Tuple[float, ...]:
    """High-precision signature for duplicate detection."""
    return tuple(round(float(config[name]), 15) for name in SEARCH_PARAMETER_ORDER)


def encode_config(config: Dict[str, Any]) -> List[float]:
    """Encode a configuration into the normalized [0,1]^5 DE search space."""
    vector: List[float] = []
    for name in SEARCH_PARAMETER_ORDER:
        spec = SEARCH_SPACE[name]
        low = float(spec["low"])
        high = float(spec["high"])
        value = float(config[name])

        if spec["distribution"] == "log":
            low_t, high_t, value_t = math.log(low), math.log(high), math.log(value)
        else:
            low_t, high_t, value_t = low, high, value

        normalized = (value_t - low_t) / (high_t - low_t)
        vector.append(float(max(0.0, min(1.0, normalized))))
    return vector


def decode_vector(vector: List[float], config_id: str) -> Dict[str, Any]:
    """Decode a repaired unit-hypercube vector into the frozen five-dimensional space."""
    if len(vector) != len(SEARCH_PARAMETER_ORDER):
        raise ValueError("DE vector length does not match the search-space dimensionality.")

    config: Dict[str, Any] = {"config_id": str(config_id)}
    for value, name in zip(vector, SEARCH_PARAMETER_ORDER):
        value = float(max(0.0, min(1.0, value)))
        spec = SEARCH_SPACE[name]
        low = float(spec["low"])
        high = float(spec["high"])

        if spec["distribution"] == "log":
            decoded = math.exp(math.log(low) + value * (math.log(high) - math.log(low)))
        else:
            decoded = low + value * (high - low)

        config[name] = float(max(low, min(high, decoded)))
    return config


def repair_unit_vector_random_reset(
    vector: List[float],
    rng: random.Random,
) -> Tuple[List[float], int]:
    """Reset each out-of-bounds DE coordinate uniformly into [0,1]."""
    repaired: List[float] = []
    repairs = 0
    for value in vector:
        value = float(value)
        if value < 0.0 or value > 1.0:
            repaired.append(float(rng.uniform(0.0, 1.0)))
            repairs += 1
        else:
            repaired.append(value)
    return repaired, int(repairs)


def rows_to_configs(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Convert successful evaluation rows back to five-dimensional configurations."""
    configs: List[Dict[str, Any]] = []
    for row in rows:
        if row.get("status") != "PASS":
            continue
        config = {"config_id": str(row["config_id"])}
        for name in SEARCH_PARAMETER_ORDER:
            config[name] = float(row[name])
        configs.append(config)
    return configs


def unique_configs(configs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Remove duplicate parameter vectors while preserving order."""
    seen = set()
    output: List[Dict[str, Any]] = []
    for config in configs:
        sig = config_signature(config)
        if sig in seen:
            continue
        seen.add(sig)
        output.append(config)
    return output


def pick_de_parents(
    *,
    target: Dict[str, Any],
    preferred_pool: List[Dict[str, Any]],
    global_pool: List[Dict[str, Any]],
    history_pool: List[Dict[str, Any]],
    rng: random.Random,
) -> Tuple[Tuple[Dict[str, Any], Dict[str, Any], Dict[str, Any]], str]:
    """
    Pick three distinct DE/rand/1 parents.

    Parent priority:
      1. intended DEHB parent pool,
      2. current global subpopulation,
      3. historical evaluated-config archive only if still necessary.

    The target vector is excluded.
    """
    target_sig = config_signature(target)

    def eligible(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        return [
            item for item in unique_configs(items)
            if config_signature(item) != target_sig
        ]

    preferred = eligible(preferred_pool)
    if len(preferred) >= 3:
        return tuple(rng.sample(preferred, 3)), "preferred_pool"

    current_global = eligible(preferred_pool + global_pool)
    if len(current_global) >= 3:
        return tuple(rng.sample(current_global, 3)), "global_population_fallback"

    historical = eligible(preferred_pool + global_pool + history_pool)
    if len(historical) >= 3:
        return tuple(rng.sample(historical, 3)), "history_archive_fallback"

    raise RuntimeError(
        "DE/rand/1 needs three distinct parents besides the target, "
        f"but only {len(historical)} are available."
    )


def make_dehb_child(
    *,
    target: Dict[str, Any],
    preferred_parent_pool: List[Dict[str, Any]],
    global_pool: List[Dict[str, Any]],
    history_pool: List[Dict[str, Any]],
    rng: random.Random,
    child_id: str,
) -> Tuple[Dict[str, Any], int, str]:
    """Generate one DE/rand/1/bin child in normalized search-space coordinates."""
    parents, parent_source = pick_de_parents(
        target=target,
        preferred_pool=preferred_parent_pool,
        global_pool=global_pool,
        history_pool=history_pool,
        rng=rng,
    )
    base, parent_1, parent_2 = parents

    base_v = encode_config(base)
    p1_v = encode_config(parent_1)
    p2_v = encode_config(parent_2)

    mutant = [
        b + DE_MUTATION_FACTOR * (p1 - p2)
        for b, p1, p2 in zip(base_v, p1_v, p2_v)
    ]
    mutant, mutation_repairs = repair_unit_vector_random_reset(mutant, rng)

    target_v = encode_config(target)
    forced_index = rng.randrange(len(target_v))
    child_v: List[float] = []

    for index, target_value in enumerate(target_v):
        if rng.random() < DE_CROSSOVER_RATE or index == forced_index:
            child_v.append(mutant[index])
        else:
            child_v.append(target_value)

    child_v, crossover_repairs = repair_unit_vector_random_reset(child_v, rng)
    child = decode_vector(child_v, child_id)
    return child, int(mutation_repairs + crossover_repairs), parent_source


def dehb_rank_key(row: Dict[str, Any]) -> Tuple[float, float, float, str]:
    """Best-first ranking: mAP50-95, then mAP50, recall, config ID."""
    return (
        -float(row["val_map50_95"]),
        -float(row["val_map50"]),
        -float(row["val_recall"]),
        str(row["config_id"]),
    )


def rank_dehb_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Rank successful DEHB evaluations using the frozen selection rule."""
    successful = [
        row for row in rows
        if row.get("status") == "PASS"
        and row.get("val_map50_95", "") != ""
    ]
    return sorted(successful, key=dehb_rank_key)


def top_rows(rows: List[Dict[str, Any]], n: int) -> List[Dict[str, Any]]:
    """Return the best n successful rows by the frozen HPO ranking."""
    return rank_dehb_rows(rows)[:max(1, int(n))]


def build_dehb_plan_rows() -> List[Dict[str, Any]]:
    """Return the exact budget-matched two-iteration DEHB evaluation plan."""
    rows = [
        # Initialization iteration.
        {"dehb_iteration": 1, "bracket_id": "DEHB1_B1", "stage_index": 1, "budget_epochs": 20, "n_configs": 2, "planned_mode": "random_initialization"},
        {"dehb_iteration": 1, "bracket_id": "DEHB1_B1", "stage_index": 2, "budget_epochs": 40, "n_configs": 1, "planned_mode": "direct_initialization_promotion"},
        {"dehb_iteration": 1, "bracket_id": "DEHB1_B2", "stage_index": 1, "budget_epochs": 40, "n_configs": 2, "planned_mode": "full_fidelity_initialization"},

        # Evolution iteration.
        {"dehb_iteration": 2, "bracket_id": "DEHB2_B1", "stage_index": 1, "budget_epochs": 20, "n_configs": 2, "planned_mode": "vanilla_de"},
        {"dehb_iteration": 2, "bracket_id": "DEHB2_B1", "stage_index": 2, "budget_epochs": 40, "n_configs": 1, "planned_mode": "lower_fidelity_guided_de"},
        {"dehb_iteration": 2, "bracket_id": "DEHB2_B2", "stage_index": 1, "budget_epochs": 40, "n_configs": 2, "planned_mode": "vanilla_de"},
    ]

    planned_evaluations = sum(int(row["n_configs"]) for row in rows)
    planned_units = sum(int(row["n_configs"]) * int(row["budget_epochs"]) for row in rows)
    planned_full = sum(
        int(row["n_configs"]) for row in rows
        if int(row["budget_epochs"]) == DEHB_MAX_EPOCHS
    )

    if planned_evaluations != EXPECTED_HPO_EVALUATIONS:
        raise RuntimeError(
            f"DEHB evaluation-count mismatch: {planned_evaluations} != {EXPECTED_HPO_EVALUATIONS}."
        )
    if planned_units != DEHB_HPO_EPOCH_UNITS:
        raise RuntimeError(
            f"DEHB epoch-unit mismatch: {planned_units} != {DEHB_HPO_EPOCH_UNITS}."
        )
    if planned_full != EXPECTED_FULL_FIDELITY_EVALUATIONS:
        raise RuntimeError(
            f"DEHB full-fidelity count mismatch: {planned_full} != "
            f"{EXPECTED_FULL_FIDELITY_EVALUATIONS}."
        )
    return rows


def update_incumbent_population(
    *,
    population: List[Dict[str, Any]],
    candidate_row: Dict[str, Any],
    capacity: int,
    target_config_id: str = "",
) -> str:
    """
    Update one fidelity-specific incumbent population.

    During DE evolution, the child replaces its target only when its validation
    mAP50-95 is at least as high. During initialization, the population is kept
    bounded by retaining the best rows.
    """
    if candidate_row.get("status") != "PASS":
        return "no_update_failed_eval"

    if target_config_id:
        for idx, incumbent in enumerate(population):
            if str(incumbent["config_id"]) == str(target_config_id):
                if float(candidate_row["val_map50_95"]) >= float(incumbent["val_map50_95"]):
                    population[idx] = candidate_row
                    return "target_replaced"
                return "target_retained"

    population.append(candidate_row)
    population[:] = top_rows(population, capacity)
    return "population_seeded_or_trimmed"


def next_target(
    population: List[Dict[str, Any]],
    pointer_state: Dict[int, int],
    budget_epochs: int,
) -> Dict[str, Any]:
    """Return a rolling target from one fidelity population."""
    if not population:
        raise RuntimeError(f"No incumbent population exists at {budget_epochs} epochs.")
    pointer = int(pointer_state.get(int(budget_epochs), 0)) % len(population)
    target_row = population[pointer]
    pointer_state[int(budget_epochs)] = (pointer + 1) % len(population)
    return target_row


## Module 8: Compact diagnostic plots

Provides one DEHB progress figure and one final target test summary figure. Together with the source versus target class distribution from Module 3, these give three focused custom plots for the complete experiment.


In [ ]:
def plot_dehb_progress(
    hpo_rows: List[Dict[str, Any]],
    output_root: Path,
) -> None:
    """Plot DEHB validation scores and the full fidelity incumbent."""
    successful = [row for row in hpo_rows if row.get("status") == "PASS"]
    if not successful:
        return

    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    x_values = [int(row["evaluation_index"]) for row in successful]
    scores = [float(row["val_map50_95"]) for row in successful]
    budgets = [int(row["budget_epochs"]) for row in successful]

    full_best = []
    incumbent = None
    for row in successful:
        if int(row["budget_epochs"]) == DEHB_MAX_EPOCHS:
            value = float(row["val_map50_95"])
            incumbent = value if incumbent is None else max(incumbent, value)
        full_best.append(np.nan if incumbent is None else incumbent)

    plt.figure(figsize=(9, 5))
    for budget in sorted(set(budgets)):
        bx = [x for x, b in zip(x_values, budgets) if b == budget]
        by = [y for y, b in zip(scores, budgets) if b == budget]
        plt.scatter(bx, by, label=f"{budget} epoch evaluation")
    plt.plot(x_values, full_best, marker="o", label="Best 40 epoch score so far")
    plt.xlabel("DEHB evaluation")
    plt.ylabel("Source validation mAP50-95")
    plt.title(f"{CROSS_YEAR_TAG}: DEHB progress")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_root / "dehb_source_validation_progress.png", dpi=180, bbox_inches="tight")
    plt.show()


def plot_final_target_test_metrics(
    final_rows: List[Dict[str, Any]],
    output_root: Path,
) -> None:
    """Plot final target test metric means with sample standard deviations."""
    successful = [row for row in final_rows if row.get("status") == "PASS"]
    if not successful:
        return

    metric_specs = [
        ("target_test_precision", "Precision"),
        ("target_test_recall", "Recall"),
        ("target_test_map50", "mAP50"),
        ("target_test_map50_95", "mAP50-95"),
    ]

    labels = []
    means = []
    sds = []
    for key, label in metric_specs:
        values = [float(row[key]) for row in successful]
        labels.append(label)
        means.append(mean(values))
        sds.append(stdev(values) if len(values) > 1 else 0.0)

    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(8, 5))
    plt.bar(labels, means, yerr=sds, capsize=4)
    plt.ylabel("Score")
    plt.ylim(0.0, 1.0)
    plt.title(f"{CROSS_YEAR_TAG}: target test mean and sample SD")
    plt.tight_layout()
    plt.savefig(output_root / "target_test_metrics_mean_sd.png", dpi=180, bbox_inches="tight")
    plt.show()


## Module 9: Target year test evaluator

Evaluates a frozen final checkpoint on the target year test split. This function is called only after DEHB selection has finished, so target test metrics cannot influence mutation, crossover, population updates, ranking, or winner selection.


In [ ]:
def evaluate_checkpoint_on_target_test_split(
    *,
    checkpoint_path: Path,
    dataset_yaml: Path,
    output_root: Path,
    run_name: str,
    expected_class_names: List[str],
    image_size: int,
    batch_size: int,
    workers: int,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Evaluate one frozen checkpoint on the target-year test split only."""
    checkpoint_path = Path(checkpoint_path)
    dataset_yaml = Path(dataset_yaml)
    output_root = Path(output_root)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    output_root.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(checkpoint_path))

    test_start = time.perf_counter()
    test_metrics = model.val(
        data=str(dataset_yaml.resolve()),
        split="test",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(output_root.resolve()),
        name=run_name,
        plots=False,
        save_json=False,
        verbose=False,
    )
    test_seconds = time.perf_counter() - test_start

    metric_data = extract_box_metrics(test_metrics, expected_class_names)

    per_class_rows = [
        {
            "class_id": row["class_id"],
            "class_name": row["class_name"],
            "target_test_map50_95": row["map50_95"],
        }
        for row in metric_data["per_class_map50_95"]
    ]

    return {
        "target_test_precision": metric_data["precision"],
        "target_test_recall": metric_data["recall"],
        "target_test_map50": metric_data["map50"],
        "target_test_map50_95": metric_data["map50_95"],
        "target_test_bottom3_mean_map50_95": metric_data["bottom3_mean_map50_95"],
        "target_test_seconds": float(test_seconds),
        "per_class_rows": per_class_rows,
    }

## Module 10: Complete cross year DEHB workflow

Runs dataset auditing, the fixed optimization split, all DEHB evaluations, the exact budget audit, full fidelity winner selection, and the three final cross year runs. It saves compact summaries and removes the large temporary YOLO and HPO folders only after all required runs succeed.


In [ ]:
def create_output_directories() -> None:
    """Create persistent and temporary experiment directories."""
    for folder in [
        PROJECT_ROOT,
        RUNS_ROOT,
        SPLITS_ROOT,
        STATS_ROOT,
        YOLO_DATA_ROOT,
        RUN_ROOT,
        HPO_TRIALS_ROOT,
        FINAL_TRAINING_ROOT,
        TARGET_TEST_EVAL_ROOT,
        RUN_STATS_ROOT,
        RUN_PLOTS_ROOT,
    ]:
        folder.mkdir(parents=True, exist_ok=True)


def run_cross_year_dehb_baseline() -> Dict[str, Any]:
    """Run the complete budget-matched 20/40-epoch cross-year DEHB experiment."""
    if not TRAIN_SOURCE_ROOT.exists():
        raise FileNotFoundError(
            f"Source dataset not found. Expected: Datasets/{TRAIN_DATASET_TAG}"
        )
    if not TEST_TARGET_ROOT.exists():
        raise FileNotFoundError(
            f"Target dataset not found. Expected: Datasets/{TEST_DATASET_TAG}"
        )

    create_output_directories()

    source_records, source_scan_stats = build_clean_records(TRAIN_SOURCE_ROOT)
    target_records, target_scan_stats = build_clean_records(TEST_TARGET_ROOT)

    source_audit = audit_records(source_records, CLASS_NAMES)
    target_audit = audit_records(target_records, CLASS_NAMES)

    if source_audit["missing_classes"]:
        raise RuntimeError(
            f"Source dataset is missing target classes: {source_audit['missing_classes']}"
        )
    if target_audit["missing_classes"]:
        raise RuntimeError(
            f"Target dataset is missing target classes: {target_audit['missing_classes']}"
        )

    save_json(source_scan_stats, STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_scan_summary.json")
    save_json(target_scan_stats, STATS_ROOT / f"{TEST_DATASET_TAG}_target_scan_summary.json")
    save_json(source_audit, STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_class_audit.json")
    save_json(target_audit, STATS_ROOT / f"{TEST_DATASET_TAG}_target_class_audit.json")

    plot_source_target_box_counts(
        source_audit=source_audit,
        target_audit=target_audit,
        output_root=RUN_PLOTS_ROOT,
    )

    workflow_start = time.perf_counter()
    device = "0" if torch.cuda.is_available() else "cpu"

    plan_csv = RUN_STATS_ROOT / "dehb_plan.csv"
    hpo_csv = RUN_STATS_ROOT / "dehb_source_validation_results.csv"
    ranked_full_csv = RUN_STATS_ROOT / "dehb_full_fidelity_ranked_results.csv"
    selected_json = RUN_STATS_ROOT / "selected_dehb_config.json"
    boundary_csv = RUN_STATS_ROOT / "selected_config_boundary_report.csv"

    final_csv = RUN_STATS_ROOT / "final_three_run_cross_year_results.csv"
    summary_csv = RUN_STATS_ROOT / "final_three_run_mean_sd_summary.csv"
    per_class_csv = RUN_STATS_ROOT / "final_three_run_target_test_per_class.csv"
    per_class_summary_csv = RUN_STATS_ROOT / "final_three_run_target_test_per_class_mean_sd.csv"
    manifest_json = RUN_STATS_ROOT / "experiment_manifest.json"

    print("=" * 100)
    print("CROSS-YEAR DEHB: TWO 20/40-EPOCH ITERATIONS")
    print("=" * 100)
    print("Direction:", CROSS_YEAR_TAG)
    print("Fitness:", "source-year validation mAP50-95 only")
    print("Optimization split seed:", OPTIMIZATION_SPLIT_SEED)
    print("Optimization train seed:", OPTIMIZATION_TRAIN_SEED)
    print("DEHB optimizer seed:", DEHB_RANDOM_SEED)
    print("Fidelities:", DEHB_MIN_EPOCHS, "->", DEHB_MAX_EPOCHS)
    print("eta:", DEHB_ETA)
    print("Iterations:", DEHB_ITERATIONS)
    print("DE mutation factor / crossover:", DE_MUTATION_FACTOR, "/", DE_CROSSOVER_RATE)
    print("Planned HPO budget:", DEHB_HPO_EPOCH_UNITS, "epoch-units")
    print("Expected HPO evaluations:", EXPECTED_HPO_EVALUATIONS)
    print("Expected full-fidelity evaluations:", EXPECTED_FULL_FIDELITY_EVALUATIONS)
    print("Workers / cache:", WORKERS, "/", CACHE_MODE)
    print("Target-year test used during HPO: NO")
    print("Device:", device)
    print("Output root:", RUN_ROOT)
    print("=" * 100)

    # --------------------------------------------------------
    # Stage 1: Prepare the single fixed cross-year HPO split.
    # --------------------------------------------------------
    optimization_cross, optimization_source, optimization_target, optimization_info = (
        make_cross_year_splits_for_seed(
            OPTIMIZATION_SPLIT_SEED, source_records, target_records
        )
    )

    optimization_dataset_root = YOLO_DATA_ROOT / f"dehb_opt_seed{OPTIMIZATION_SPLIT_SEED}"
    optimization_manifest_root = (
        SPLITS_ROOT / CROSS_YEAR_TAG / f"dehb_opt_seed{OPTIMIZATION_SPLIT_SEED}"
    )

    optimization_yaml, optimization_coverage = prepare_cross_year_yolo_dataset(
        cross_year_splits=optimization_cross,
        source_splits=optimization_source,
        target_splits=optimization_target,
        split_info=optimization_info,
        output_root=optimization_dataset_root,
        manifest_root=optimization_manifest_root,
        image_mode=IMAGE_PLACEMENT_MODE,
        seed=OPTIMIZATION_SPLIT_SEED,
    )

    # --------------------------------------------------------
    # Stage 2: Save and audit the exact DEHB plan.
    # --------------------------------------------------------
    plan_rows = build_dehb_plan_rows()
    save_rows_to_csv_safe(plan_rows, plan_csv)

    print("\n" + "=" * 100)
    print("DEHB PLAN")
    print("=" * 100)
    for row in plan_rows:
        print(
            f"iteration={row['dehb_iteration']} | "
            f"{row['bracket_id']} | stage={row['stage_index']} | "
            f"{row['n_configs']} x {row['budget_epochs']} epochs | "
            f"{row['planned_mode']}"
        )
    print("Planned HPO epoch-units:", DEHB_HPO_EPOCH_UNITS)
    print("=" * 100)

    # --------------------------------------------------------
    # Stage 3: DEHB HPO.
    # --------------------------------------------------------
    selection_start = time.perf_counter()
    rng = random.Random(DEHB_RANDOM_SEED)

    hpo_rows: List[Dict[str, Any]] = []
    history_rows: List[Dict[str, Any]] = []

    # Paper-style maximum HB allocation per fidelity for this tiny schedule.
    population_capacities = {
        SCREEN_EPOCHS: 2,
        FULL_EPOCHS: 2,
    }
    populations: Dict[int, List[Dict[str, Any]]] = {
        SCREEN_EPOCHS: [],
        FULL_EPOCHS: [],
    }
    pointer_state = {
        SCREEN_EPOCHS: 0,
        FULL_EPOCHS: 0,
    }

    config_counter = 0
    evaluation_index = 0

    def evaluate_config(
        *,
        config: Dict[str, Any],
        dehb_iteration: int,
        bracket_id: str,
        stage_index: int,
        budget_epochs: int,
        generation_mode: str,
        mutation_type: str,
        target_config_id: str = "",
        parent_source: str = "",
        boundary_repair_count: int = 0,
    ) -> Dict[str, Any]:
        nonlocal evaluation_index
        evaluation_index += 1

        warmup_epochs, close_mosaic = get_fidelity_controls(budget_epochs)
        evaluation_start = time.perf_counter()

        try:
            trial_result = train_and_validate_configuration(
                learning_rate=float(config["learning_rate"]),
                lrf=float(config["lrf"]),
                momentum=float(config["momentum"]),
                weight_decay=float(config["weight_decay"]),
                scale=float(config["scale"]),
                dataset_yaml=optimization_yaml,
                project_dir=(
                    HPO_TRIALS_ROOT
                    / f"iter{dehb_iteration}"
                    / bracket_id
                    / f"stage{stage_index}"
                ),
                model_name=MODEL_NAME,
                epochs=int(budget_epochs),
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                patience=PATIENCE,
                workers=WORKERS,
                seed=OPTIMIZATION_TRAIN_SEED,
                warmup_epochs=float(warmup_epochs),
                close_mosaic=int(close_mosaic),
                fixed_training_args=FIXED_TRAINING_ARGS,
                expected_class_names=CLASS_NAMES,
                strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
                fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE,
                device=device,
            )

            evaluation_wall_seconds = time.perf_counter() - evaluation_start
            metrics = trial_result["metrics"]

            row = {
                "status": "PASS",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                "evaluation_index": int(evaluation_index),
                "dehb_iteration": int(dehb_iteration),
                "bracket_id": str(bracket_id),
                "stage_index": int(stage_index),
                "budget_epochs": int(budget_epochs),
                "config_id": str(config["config_id"]),
                "target_config_id": str(target_config_id),
                "generation_mode": str(generation_mode),
                "mutation_type": str(mutation_type),
                "parent_source": str(parent_source),
                "boundary_repair_count": int(boundary_repair_count),
                "learning_rate": float(config["learning_rate"]),
                "lrf": float(config["lrf"]),
                "momentum": float(config["momentum"]),
                "weight_decay": float(config["weight_decay"]),
                "scale": float(config["scale"]),
                "warmup_epochs": float(warmup_epochs),
                "close_mosaic": int(close_mosaic),
                "val_precision": float(metrics["precision"]),
                "val_recall": float(metrics["recall"]),
                "val_map50": float(metrics["map50"]),
                "val_map50_95": float(metrics["map50_95"]),
                "val_bottom3_mean_map50_95": float(metrics["bottom3_mean_map50_95"]),
                "validation_fitness": float(metrics["map50_95"]),
                "source_train_count": int(optimization_info["effective_cross_year_counts"]["train"]),
                "source_valid_count": int(optimization_info["effective_cross_year_counts"]["valid"]),
                "target_test_count": int(optimization_info["effective_cross_year_counts"]["test"]),
                "subsample_enabled": bool(optimization_info["subsample_info"]["enabled"]),
                "subsample_seed": optimization_info["subsample_info"]["subsample_seed"],
                "fitness_alignment_status": trial_result["fitness_alignment"]["status"],
                "training_seconds": float(trial_result["timing"]["training_seconds"]),
                "validation_seconds": float(trial_result["timing"]["validation_seconds"]),
                "evaluation_wall_seconds": float(evaluation_wall_seconds),
                "best_checkpoint": relative_output_path(
                    Path(trial_result["paths"]["best_checkpoint"]), WORKSPACE_ROOT
                ),
                "population_update": "",
                "error_message": "",
            }

        except Exception as error:
            evaluation_wall_seconds = time.perf_counter() - evaluation_start
            print("\nERROR during DEHB evaluation:")
            print(privacy_safe_text(error, WORKSPACE_ROOT))

            row = {
                "status": "FAIL",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                "evaluation_index": int(evaluation_index),
                "dehb_iteration": int(dehb_iteration),
                "bracket_id": str(bracket_id),
                "stage_index": int(stage_index),
                "budget_epochs": int(budget_epochs),
                "config_id": str(config.get("config_id", "")),
                "target_config_id": str(target_config_id),
                "generation_mode": str(generation_mode),
                "mutation_type": str(mutation_type),
                "parent_source": str(parent_source),
                "boundary_repair_count": int(boundary_repair_count),
                "learning_rate": config.get("learning_rate", ""),
                "lrf": config.get("lrf", ""),
                "momentum": config.get("momentum", ""),
                "weight_decay": config.get("weight_decay", ""),
                "scale": config.get("scale", ""),
                "warmup_epochs": float(warmup_epochs),
                "close_mosaic": int(close_mosaic),
                "val_precision": "",
                "val_recall": "",
                "val_map50": "",
                "val_map50_95": "",
                "val_bottom3_mean_map50_95": "",
                "validation_fitness": "",
                "source_train_count": int(optimization_info["effective_cross_year_counts"]["train"]),
                "source_valid_count": int(optimization_info["effective_cross_year_counts"]["valid"]),
                "target_test_count": int(optimization_info["effective_cross_year_counts"]["test"]),
                "subsample_enabled": bool(optimization_info["subsample_info"]["enabled"]),
                "subsample_seed": optimization_info["subsample_info"]["subsample_seed"],
                "fitness_alignment_status": "",
                "training_seconds": "",
                "validation_seconds": "",
                "evaluation_wall_seconds": float(evaluation_wall_seconds),
                "best_checkpoint": "",
                "population_update": "",
                "error_message": privacy_safe_text(error, WORKSPACE_ROOT),
            }

        hpo_rows.append(row)
        if row["status"] == "PASS":
            history_rows.append(row)

        save_rows_to_csv_safe(hpo_rows, hpo_csv)

        if row["status"] != "PASS" and REQUIRE_FULL_HPO_BUDGET:
            raise RuntimeError(
                f"DEHB evaluation {evaluation_index} failed. "
                "The run is stopped rather than reporting an under-budget comparison."
            )

        return row

    # ========================================================
    # Iteration 1: initialize the small fidelity populations.
    # ========================================================
    print("\n" + "#" * 100)
    print("DEHB ITERATION 1/2 : INITIALIZATION")
    print("#" * 100)

    # Bracket 1, 20 epochs: two random configurations.
    init_low_rows: List[Dict[str, Any]] = []
    for _ in range(2):
        config_counter += 1
        config = sample_dehb_config(rng=rng, config_id=f"C{config_counter:03d}")
        row = evaluate_config(
            config=config,
            dehb_iteration=1,
            bracket_id="DEHB1_B1",
            stage_index=1,
            budget_epochs=SCREEN_EPOCHS,
            generation_mode="random_initialization",
            mutation_type="none",
        )
        init_low_rows.append(row)
        row["population_update"] = update_incumbent_population(
            population=populations[SCREEN_EPOCHS],
            candidate_row=row,
            capacity=population_capacities[SCREEN_EPOCHS],
        )
        save_rows_to_csv_safe(hpo_rows, hpo_csv)

    # Bracket 1, 40 epochs: direct initialization promotion of the best 20-epoch config.
    promoted_low = rank_dehb_rows(init_low_rows)[0]
    promoted_config = rows_to_configs([promoted_low])[0]
    config_counter += 1
    promoted_config["config_id"] = f"P{config_counter:03d}"

    promoted_row = evaluate_config(
        config=promoted_config,
        dehb_iteration=1,
        bracket_id="DEHB1_B1",
        stage_index=2,
        budget_epochs=FULL_EPOCHS,
        generation_mode="direct_initialization_promotion",
        mutation_type="none",
        parent_source=f"promoted_from:{promoted_low['config_id']}",
    )
    promoted_row["population_update"] = update_incumbent_population(
        population=populations[FULL_EPOCHS],
        candidate_row=promoted_row,
        capacity=population_capacities[FULL_EPOCHS],
    )
    save_rows_to_csv_safe(hpo_rows, hpo_csv)

    # Bracket 2, 40 epochs: two additional full-fidelity initialization configs.
    for _ in range(2):
        config_counter += 1
        config = sample_dehb_config(rng=rng, config_id=f"C{config_counter:03d}")
        row = evaluate_config(
            config=config,
            dehb_iteration=1,
            bracket_id="DEHB1_B2",
            stage_index=1,
            budget_epochs=FULL_EPOCHS,
            generation_mode="full_fidelity_initialization",
            mutation_type="none",
        )
        row["population_update"] = update_incumbent_population(
            population=populations[FULL_EPOCHS],
            candidate_row=row,
            capacity=population_capacities[FULL_EPOCHS],
        )
        save_rows_to_csv_safe(hpo_rows, hpo_csv)

    if len(populations[SCREEN_EPOCHS]) != 2 or len(populations[FULL_EPOCHS]) != 2:
        raise RuntimeError(
            "DEHB initialization did not create the expected 20- and 40-epoch populations."
        )

    # ========================================================
    # Iteration 2: genuine DE evolution.
    # ========================================================
    print("\n" + "#" * 100)
    print("DEHB ITERATION 2/2 : DIFFERENTIAL EVOLUTION")
    print("#" * 100)

    evolved_low_rows: List[Dict[str, Any]] = []

    # Bracket 1, 20 epochs: two vanilla-DE challengers.
    for _ in range(2):
        target_row = next_target(
            population=populations[SCREEN_EPOCHS],
            pointer_state=pointer_state,
            budget_epochs=SCREEN_EPOCHS,
        )
        target_config = rows_to_configs([target_row])[0]

        current_global_configs = rows_to_configs(
            populations[SCREEN_EPOCHS] + populations[FULL_EPOCHS]
        )
        history_configs = rows_to_configs(history_rows)

        config_counter += 1
        child, repairs, parent_source = make_dehb_child(
            target=target_config,
            preferred_parent_pool=rows_to_configs(populations[SCREEN_EPOCHS]),
            global_pool=current_global_configs,
            history_pool=history_configs,
            rng=rng,
            child_id=f"DE{config_counter:03d}",
        )

        row = evaluate_config(
            config=child,
            dehb_iteration=2,
            bracket_id="DEHB2_B1",
            stage_index=1,
            budget_epochs=SCREEN_EPOCHS,
            generation_mode="de_trial_generation",
            mutation_type="vanilla_de",
            target_config_id=str(target_row["config_id"]),
            parent_source=parent_source,
            boundary_repair_count=repairs,
        )
        row["population_update"] = update_incumbent_population(
            population=populations[SCREEN_EPOCHS],
            candidate_row=row,
            capacity=population_capacities[SCREEN_EPOCHS],
            target_config_id=str(target_row["config_id"]),
        )
        evolved_low_rows.append(row)
        save_rows_to_csv_safe(hpo_rows, hpo_csv)

    # Bracket 1, 40 epochs: lower-fidelity-guided DE.
    lower_parent_rows = top_rows(evolved_low_rows, 1)
    target_row = next_target(
        population=populations[FULL_EPOCHS],
        pointer_state=pointer_state,
        budget_epochs=FULL_EPOCHS,
    )
    target_config = rows_to_configs([target_row])[0]

    config_counter += 1
    child, repairs, parent_source = make_dehb_child(
        target=target_config,
        preferred_parent_pool=rows_to_configs(lower_parent_rows),
        global_pool=rows_to_configs(
            populations[SCREEN_EPOCHS] + populations[FULL_EPOCHS]
        ),
        history_pool=rows_to_configs(history_rows),
        rng=rng,
        child_id=f"DE{config_counter:03d}",
    )

    row = evaluate_config(
        config=child,
        dehb_iteration=2,
        bracket_id="DEHB2_B1",
        stage_index=2,
        budget_epochs=FULL_EPOCHS,
        generation_mode="de_trial_generation",
        mutation_type="lower_fidelity_guided_de",
        target_config_id=str(target_row["config_id"]),
        parent_source=parent_source,
        boundary_repair_count=repairs,
    )
    row["population_update"] = update_incumbent_population(
        population=populations[FULL_EPOCHS],
        candidate_row=row,
        capacity=population_capacities[FULL_EPOCHS],
        target_config_id=str(target_row["config_id"]),
    )
    save_rows_to_csv_safe(hpo_rows, hpo_csv)

    # Bracket 2, 40 epochs: two vanilla-DE challengers.
    for _ in range(2):
        target_row = next_target(
            population=populations[FULL_EPOCHS],
            pointer_state=pointer_state,
            budget_epochs=FULL_EPOCHS,
        )
        target_config = rows_to_configs([target_row])[0]

        config_counter += 1
        child, repairs, parent_source = make_dehb_child(
            target=target_config,
            preferred_parent_pool=rows_to_configs(populations[FULL_EPOCHS]),
            global_pool=rows_to_configs(
                populations[SCREEN_EPOCHS] + populations[FULL_EPOCHS]
            ),
            history_pool=rows_to_configs(history_rows),
            rng=rng,
            child_id=f"DE{config_counter:03d}",
        )

        row = evaluate_config(
            config=child,
            dehb_iteration=2,
            bracket_id="DEHB2_B2",
            stage_index=1,
            budget_epochs=FULL_EPOCHS,
            generation_mode="de_trial_generation",
            mutation_type="vanilla_de",
            target_config_id=str(target_row["config_id"]),
            parent_source=parent_source,
            boundary_repair_count=repairs,
        )
        row["population_update"] = update_incumbent_population(
            population=populations[FULL_EPOCHS],
            candidate_row=row,
            capacity=population_capacities[FULL_EPOCHS],
            target_config_id=str(target_row["config_id"]),
        )
        save_rows_to_csv_safe(hpo_rows, hpo_csv)

    selection_seconds = time.perf_counter() - selection_start

    # --------------------------------------------------------
    # Stage 4: Exact-budget audit and full-fidelity-only winner.
    # --------------------------------------------------------
    successful_hpo = [row for row in hpo_rows if row["status"] == "PASS"]
    actual_successful_epoch_units = sum(int(row["budget_epochs"]) for row in successful_hpo)

    if REQUIRE_FULL_HPO_BUDGET:
        if len(successful_hpo) != EXPECTED_HPO_EVALUATIONS:
            raise RuntimeError(
                f"Expected {EXPECTED_HPO_EVALUATIONS} successful HPO evaluations, "
                f"got {len(successful_hpo)}."
            )
        if actual_successful_epoch_units != DEHB_HPO_EPOCH_UNITS:
            raise RuntimeError(
                f"Expected {DEHB_HPO_EPOCH_UNITS} HPO epoch-units, "
                f"got {actual_successful_epoch_units}."
            )

    full_fidelity_rows = [
        row for row in successful_hpo
        if int(row["budget_epochs"]) == DEHB_MAX_EPOCHS
    ]

    if REQUIRE_FULL_HPO_BUDGET and len(full_fidelity_rows) != EXPECTED_FULL_FIDELITY_EVALUATIONS:
        raise RuntimeError(
            f"Expected {EXPECTED_FULL_FIDELITY_EVALUATIONS} successful 40-epoch evaluations, "
            f"got {len(full_fidelity_rows)}."
        )

    genuine_de_rows = [
        row for row in successful_hpo
        if row["generation_mode"] == "de_trial_generation"
    ]
    if len(genuine_de_rows) != 5:
        raise RuntimeError(
            f"Expected 5 DE-generated evaluations in iteration 2, got {len(genuine_de_rows)}."
        )

    ranked_full = rank_dehb_rows(full_fidelity_rows)
    for rank, row in enumerate(ranked_full, start=1):
        row["full_fidelity_rank"] = int(rank)
    save_rows_to_csv_safe(ranked_full, ranked_full_csv)

    winner = ranked_full[0]
    selected_config = {
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "method": "budget-matched sequential DEHB",
        "fitness": "source-year validation mAP50-95",
        "target_test_used_for_selection": False,
        "selected_config_id": str(winner["config_id"]),
        "selected_dehb_iteration": int(winner["dehb_iteration"]),
        "selected_bracket_id": str(winner["bracket_id"]),
        "selected_budget_epochs": int(winner["budget_epochs"]),
        "selected_source_val_map50_95": float(winner["val_map50_95"]),
        "selected_source_val_map50": float(winner["val_map50"]),
        "selected_source_val_recall": float(winner["val_recall"]),
        "selected_generation_mode": str(winner["generation_mode"]),
        "selected_mutation_type": str(winner["mutation_type"]),
        "learning_rate": float(winner["learning_rate"]),
        "lrf": float(winner["lrf"]),
        "momentum": float(winner["momentum"]),
        "weight_decay": float(winner["weight_decay"]),
        "scale": float(winner["scale"]),
        "screen_epochs": SCREEN_EPOCHS,
        "screen_warmup_epochs": SCREEN_WARMUP_EPOCHS,
        "screen_close_mosaic": SCREEN_CLOSE_MOSAIC,
        "full_epochs": FULL_EPOCHS,
        "full_warmup_epochs": FULL_WARMUP_EPOCHS,
        "full_close_mosaic": FULL_CLOSE_MOSAIC,
        "dehb_eta": DEHB_ETA,
        "dehb_iterations": DEHB_ITERATIONS,
        "dehb_random_seed": DEHB_RANDOM_SEED,
        "de_mutation_factor": DE_MUTATION_FACTOR,
        "de_crossover_rate": DE_CROSSOVER_RATE,
        "hpo_epoch_units": DEHB_HPO_EPOCH_UNITS,
        "actual_successful_hpo_epoch_units": actual_successful_epoch_units,
        "hpo_successful_evaluations": len(successful_hpo),
        "hpo_full_fidelity_evaluations": len(full_fidelity_rows),
        "de_generated_evaluations": len(genuine_de_rows),
        "optimization_split_seed": OPTIMIZATION_SPLIT_SEED,
        "optimization_train_seed": OPTIMIZATION_TRAIN_SEED,
        "selection_stage_wall_seconds": float(selection_seconds),
        "search_space": SEARCH_SPACE,
        "fixed_training_args": FIXED_TRAINING_ARGS,
    }
    save_json(selected_config, selected_json)

    boundary_report = boundary_proximity_report(
        selected_config=selected_config,
        search_space=SEARCH_SPACE,
        parameter_order=SEARCH_PARAMETER_ORDER,
    )
    save_rows_to_csv_safe(boundary_report, boundary_csv)

    plot_dehb_progress(hpo_rows, RUN_PLOTS_ROOT)

    print("\n" + "=" * 100)
    print("DEHB HPO SELECTION COMPLETE")
    print("=" * 100)
    print("Selected config:", selected_config["selected_config_id"])
    print("Selected iteration:", selected_config["selected_dehb_iteration"])
    print("Selected bracket:", selected_config["selected_bracket_id"])
    print("Selected source validation mAP50-95:", f"{selected_config['selected_source_val_map50_95']:.6f}")
    print("Successful HPO evaluations:", len(successful_hpo), "/", EXPECTED_HPO_EVALUATIONS)
    print("Successful HPO epoch-units:", actual_successful_epoch_units)
    print("DE-generated evaluations:", len(genuine_de_rows))
    print("Selection-stage wall time (s):", f"{selection_seconds:.4f}")
    print("Target-year test used for selection: NO")
    print("=" * 100)

    # --------------------------------------------------------
    # Stage 5: Freeze winner and perform final three-seed cross-year evaluation.
    # --------------------------------------------------------
    final_start = time.perf_counter()
    final_rows: List[Dict[str, Any]] = []
    all_per_class_rows: List[Dict[str, Any]] = []

    for seed in FINAL_EVALUATION_SEEDS:
        run_start = time.perf_counter()

        print("\n\n" + "#" * 100)
        print(f"FINAL CROSS-YEAR DEHB EVALUATION | {CROSS_YEAR_TAG} | SEED {seed}")
        print("#" * 100)

        try:
            final_cross, final_source, final_target, final_info = (
                make_cross_year_splits_for_seed(seed, source_records, target_records)
            )

            final_dataset_root = YOLO_DATA_ROOT / f"final_seed{seed}"
            final_manifest_root = SPLITS_ROOT / CROSS_YEAR_TAG / f"final_seed{seed}"

            final_yaml, final_coverage = prepare_cross_year_yolo_dataset(
                cross_year_splits=final_cross,
                source_splits=final_source,
                target_splits=final_target,
                split_info=final_info,
                output_root=final_dataset_root,
                manifest_root=final_manifest_root,
                image_mode=IMAGE_PLACEMENT_MODE,
                seed=seed,
            )

            final_train_result = train_and_validate_configuration(
                learning_rate=selected_config["learning_rate"],
                lrf=selected_config["lrf"],
                momentum=selected_config["momentum"],
                weight_decay=selected_config["weight_decay"],
                scale=selected_config["scale"],
                dataset_yaml=final_yaml,
                project_dir=FINAL_TRAINING_ROOT / f"seed{seed}",
                model_name=MODEL_NAME,
                epochs=FINAL_EPOCHS,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                patience=PATIENCE,
                workers=WORKERS,
                seed=seed,
                warmup_epochs=FINAL_WARMUP_EPOCHS,
                close_mosaic=FINAL_CLOSE_MOSAIC,
                fixed_training_args=FIXED_TRAINING_ARGS,
                expected_class_names=CLASS_NAMES,
                strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
                fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE,
                device=device,
            )

            target_test_result = evaluate_checkpoint_on_target_test_split(
                checkpoint_path=Path(final_train_result["paths"]["best_checkpoint"]),
                dataset_yaml=final_yaml,
                output_root=TARGET_TEST_EVAL_ROOT / f"seed{seed}",
                run_name=f"target_test_seed{seed}",
                expected_class_names=CLASS_NAMES,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                workers=WORKERS,
                device=device,
            )

            final_run_wall_seconds = time.perf_counter() - run_start
            source_metrics = final_train_result["metrics"]

            final_row = {
                "status": "PASS",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                "split_seed": int(seed),
                "selected_config_id": selected_config["selected_config_id"],
                "selected_dehb_iteration": selected_config["selected_dehb_iteration"],
                "selected_bracket_id": selected_config["selected_bracket_id"],
                "selected_source_val_map50_95": selected_config["selected_source_val_map50_95"],
                "learning_rate": selected_config["learning_rate"],
                "lrf": selected_config["lrf"],
                "momentum": selected_config["momentum"],
                "weight_decay": selected_config["weight_decay"],
                "scale": selected_config["scale"],
                "final_epochs": FINAL_EPOCHS,
                "warmup_epochs": FINAL_WARMUP_EPOCHS,
                "close_mosaic": FINAL_CLOSE_MOSAIC,
                "final_source_val_precision": float(source_metrics["precision"]),
                "final_source_val_recall": float(source_metrics["recall"]),
                "final_source_val_map50": float(source_metrics["map50"]),
                "final_source_val_map50_95": float(source_metrics["map50_95"]),
                "final_source_val_bottom3_mean_map50_95": float(
                    source_metrics["bottom3_mean_map50_95"]
                ),
                "target_test_precision": float(target_test_result["target_test_precision"]),
                "target_test_recall": float(target_test_result["target_test_recall"]),
                "target_test_map50": float(target_test_result["target_test_map50"]),
                "target_test_map50_95": float(target_test_result["target_test_map50_95"]),
                "target_test_bottom3_mean_map50_95": float(
                    target_test_result["target_test_bottom3_mean_map50_95"]
                ),
                "source_train_count": int(final_info["effective_cross_year_counts"]["train"]),
                "source_valid_count": int(final_info["effective_cross_year_counts"]["valid"]),
                "target_test_count": int(final_info["effective_cross_year_counts"]["test"]),
                "subsample_enabled": bool(final_info["subsample_info"]["enabled"]),
                "subsample_seed": final_info["subsample_info"]["subsample_seed"],
                "final_training_seconds": float(final_train_result["timing"]["training_seconds"]),
                "final_source_validation_seconds": float(
                    final_train_result["timing"]["validation_seconds"]
                ),
                "target_test_seconds": float(target_test_result["target_test_seconds"]),
                "final_run_wall_seconds": float(final_run_wall_seconds),
                "best_checkpoint": relative_output_path(
                    Path(final_train_result["paths"]["best_checkpoint"]), WORKSPACE_ROOT
                ),
                "dataset_yaml": relative_output_path(final_yaml, WORKSPACE_ROOT),
                "error_message": "",
            }

            for class_row in target_test_result["per_class_rows"]:
                all_per_class_rows.append({
                    "cross_year_tag": CROSS_YEAR_TAG,
                    "train_year": TRAIN_YEAR,
                    "test_year": TEST_YEAR,
                    "split_seed": int(seed),
                    "class_id": class_row["class_id"],
                    "class_name": class_row["class_name"],
                    "target_test_map50_95": class_row["target_test_map50_95"],
                })

        except Exception as error:
            final_run_wall_seconds = time.perf_counter() - run_start
            print("\nERROR during final cross-year DEHB evaluation:")
            print(privacy_safe_text(error, WORKSPACE_ROOT))

            final_row = {
                "status": "FAIL",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                "split_seed": int(seed),
                "selected_config_id": selected_config["selected_config_id"],
                "selected_dehb_iteration": selected_config["selected_dehb_iteration"],
                "selected_bracket_id": selected_config["selected_bracket_id"],
                "selected_source_val_map50_95": selected_config["selected_source_val_map50_95"],
                "learning_rate": selected_config["learning_rate"],
                "lrf": selected_config["lrf"],
                "momentum": selected_config["momentum"],
                "weight_decay": selected_config["weight_decay"],
                "scale": selected_config["scale"],
                "final_epochs": FINAL_EPOCHS,
                "warmup_epochs": FINAL_WARMUP_EPOCHS,
                "close_mosaic": FINAL_CLOSE_MOSAIC,
                "final_source_val_precision": "",
                "final_source_val_recall": "",
                "final_source_val_map50": "",
                "final_source_val_map50_95": "",
                "final_source_val_bottom3_mean_map50_95": "",
                "target_test_precision": "",
                "target_test_recall": "",
                "target_test_map50": "",
                "target_test_map50_95": "",
                "target_test_bottom3_mean_map50_95": "",
                "source_train_count": "",
                "source_valid_count": "",
                "target_test_count": "",
                "subsample_enabled": "",
                "subsample_seed": "",
                "final_training_seconds": "",
                "final_source_validation_seconds": "",
                "target_test_seconds": "",
                "final_run_wall_seconds": float(final_run_wall_seconds),
                "best_checkpoint": "",
                "dataset_yaml": "",
                "error_message": privacy_safe_text(error, WORKSPACE_ROOT),
            }

        final_rows.append(final_row)
        save_rows_to_csv_safe(final_rows, final_csv)
        save_rows_to_csv_safe(all_per_class_rows, per_class_csv)

    final_seconds = time.perf_counter() - final_start
    successful_final = [row for row in final_rows if row["status"] == "PASS"]

    if REQUIRE_ALL_FINAL_RUNS and len(successful_final) != len(FINAL_EVALUATION_SEEDS):
        raise RuntimeError(
            f"Expected {len(FINAL_EVALUATION_SEEDS)} successful final runs, "
            f"got {len(successful_final)}."
        )

    # --------------------------------------------------------
    # Stage 6: Summaries, manifest, and final compact plot.
    # --------------------------------------------------------
    summary_metric_names = [
        "final_source_val_precision",
        "final_source_val_recall",
        "final_source_val_map50",
        "final_source_val_map50_95",
        "final_source_val_bottom3_mean_map50_95",
        "target_test_precision",
        "target_test_recall",
        "target_test_map50",
        "target_test_map50_95",
        "target_test_bottom3_mean_map50_95",
        "final_training_seconds",
        "final_source_validation_seconds",
        "target_test_seconds",
        "final_run_wall_seconds",
    ]

    summary_rows = summarize_metric_mean_sd(
        successful_final,
        summary_metric_names,
    )

    total_workflow_seconds = time.perf_counter() - workflow_start

    def add_single_value_summary(metric: str, value: float) -> None:
        summary_rows.append({
            "metric": metric,
            "n": 1,
            "mean": float(value),
            "sd": "",
            "min": float(value),
            "max": float(value),
        })

    add_single_value_summary("hpo_epoch_units", DEHB_HPO_EPOCH_UNITS)
    add_single_value_summary("actual_successful_hpo_epoch_units", actual_successful_epoch_units)
    add_single_value_summary("hpo_successful_evaluations", len(successful_hpo))
    add_single_value_summary("hpo_full_fidelity_evaluations", len(full_fidelity_rows))
    add_single_value_summary("de_generated_evaluations", len(genuine_de_rows))
    add_single_value_summary("final_evaluation_epoch_units", FINAL_EVALUATION_EPOCH_UNITS)
    add_single_value_summary("selection_stage_wall_seconds", selection_seconds)
    add_single_value_summary("final_stage_wall_seconds", final_seconds)
    add_single_value_summary("total_workflow_wall_seconds", total_workflow_seconds)

    save_rows_to_csv_safe(summary_rows, summary_csv)

    per_class_summary_rows = summarize_per_class_mean_sd(all_per_class_rows)
    save_rows_to_csv_safe(per_class_summary_rows, per_class_summary_csv)

    manifest = {
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "method": "DEHB",
        "model_name": MODEL_NAME,
        "fitness": "source-year validation mAP50-95 only",
        "target_test_used_during_hpo": False,
        "search_space_version": SEARCH_SPACE_VERSION,
        "search_space": SEARCH_SPACE,
        "fixed_training_args": FIXED_TRAINING_ARGS,
        "runtime": {
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "workers": WORKERS,
            "patience": PATIENCE,
            "cache": CACHE_MODE,
            "deterministic": True,
        },
        "subsampling_protocol": {
            "year2021_source_only": SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE,
            "target_mode": SUBSAMPLE_TARGET_MODE,
            "seed_offset": SUBSAMPLE_SEED_OFFSET,
            "primary_label_rule": (
                "most frequent class in each image; ties follow CLASS_NAMES order"
            ),
            "optimization_split_info": optimization_info,
        },
        "dehb": {
            "min_epochs": DEHB_MIN_EPOCHS,
            "max_epochs": DEHB_MAX_EPOCHS,
            "eta": DEHB_ETA,
            "iterations": DEHB_ITERATIONS,
            "optimizer_seed": DEHB_RANDOM_SEED,
            "mutation_factor": DE_MUTATION_FACTOR,
            "crossover_rate": DE_CROSSOVER_RATE,
            "screen_warmup_epochs": SCREEN_WARMUP_EPOCHS,
            "screen_close_mosaic": SCREEN_CLOSE_MOSAIC,
            "full_warmup_epochs": FULL_WARMUP_EPOCHS,
            "full_close_mosaic": FULL_CLOSE_MOSAIC,
            "planned_hpo_epoch_units": DEHB_HPO_EPOCH_UNITS,
            "actual_successful_hpo_epoch_units": actual_successful_epoch_units,
            "expected_evaluations": EXPECTED_HPO_EVALUATIONS,
            "expected_full_fidelity_evaluations": EXPECTED_FULL_FIDELITY_EVALUATIONS,
            "de_generated_evaluations": len(genuine_de_rows),
            "parent_fallback_rule": (
                "preferred fidelity population, then current global populations, "
                "then unique historical evaluated configurations if required for DE/rand/1"
            ),
        },
        "seeds": {
            "optimization_split_seed": OPTIMIZATION_SPLIT_SEED,
            "optimization_train_seed": OPTIMIZATION_TRAIN_SEED,
            "final_evaluation_seeds": FINAL_EVALUATION_SEEDS,
        },
        "selected_config": selected_config,
        "source_scan_stats": source_scan_stats,
        "target_scan_stats": target_scan_stats,
        "source_audit": source_audit,
        "target_audit": target_audit,
        "optimization_split_coverage": optimization_coverage,
        "selection_stage_wall_seconds": float(selection_seconds),
        "final_stage_wall_seconds": float(final_seconds),
        "total_workflow_wall_seconds": float(total_workflow_seconds),
        "temporary_cleanup": {
            "yolo_data": "removed after all required runs succeed",
            "hpo_trials": "removed after all required runs succeed",
        },
        "software": {
            "python": sys.version,
            "ultralytics": ultralytics.__version__,
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda,
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        },
        "output_root": relative_output_path(RUN_ROOT, WORKSPACE_ROOT),
    }
    save_json(manifest, manifest_json)

    plot_final_target_test_metrics(final_rows, RUN_PLOTS_ROOT)

    print("\n\n" + "=" * 100)
    print("CROSS-YEAR DEHB COMPLETE")
    print("=" * 100)
    print("Direction:", CROSS_YEAR_TAG)
    print("Selected config:", selected_config["selected_config_id"])
    print("Selected iteration:", selected_config["selected_dehb_iteration"])
    print("Selected bracket:", selected_config["selected_bracket_id"])
    print(
        "Selected 40-epoch source validation mAP50-95:",
        f"{selected_config['selected_source_val_map50_95']:.6f}",
    )
    print("Successful HPO evaluations:", len(successful_hpo), "/", EXPECTED_HPO_EVALUATIONS)
    print("Full-fidelity evaluations:", len(full_fidelity_rows), "/", EXPECTED_FULL_FIDELITY_EVALUATIONS)
    print("DE-generated evaluations:", len(genuine_de_rows))
    print("HPO epoch-units:", actual_successful_epoch_units, "/", DEHB_HPO_EPOCH_UNITS)
    print("Optimization/selection time (s):", f"{selection_seconds:.4f}")
    print("Successful final runs:", len(successful_final), "/", len(FINAL_EVALUATION_SEEDS))
    print("Target-year test data was used only after the DEHB configuration was frozen.")

    if successful_final:
        print("Mean and sample SD final statistics:")
        for metric_name in summary_metric_names:
            values = [
                float(row[metric_name])
                for row in successful_final
                if row.get(metric_name, "") != ""
            ]
            if not values:
                continue
            if len(values) == 1:
                print(f"{metric_name}: {values[0]:.4f}")
            else:
                print(f"{metric_name}: {mean(values):.4f} +/- {stdev(values):.4f}")

    print("selection_stage_wall_seconds:", f"{selection_seconds:.4f}")
    print("final_stage_wall_seconds:", f"{final_seconds:.4f}")
    print("total_workflow_wall_seconds:", f"{total_workflow_seconds:.4f}")

    print("\nSaved files:")
    for path in [
        plan_csv,
        hpo_csv,
        ranked_full_csv,
        selected_json,
        boundary_csv,
        final_csv,
        summary_csv,
        per_class_csv,
        per_class_summary_csv,
        manifest_json,
    ]:
        print("*", relative_output_path(path, WORKSPACE_ROOT))
    print("=" * 100)

    # Remove large temporary folders only after every required HPO and final run succeeds.
    if YOLO_DATA_ROOT.exists():
        shutil.rmtree(YOLO_DATA_ROOT)

    if HPO_TRIALS_ROOT.exists():
        shutil.rmtree(HPO_TRIALS_ROOT)

    print("\nCleanup completed:")
    print("Removed:", relative_output_path(YOLO_DATA_ROOT, WORKSPACE_ROOT))
    print("Removed:", relative_output_path(HPO_TRIALS_ROOT, WORKSPACE_ROOT))

    return {
        "selected_config": selected_config,
        "hpo_rows": hpo_rows,
        "ranked_full_fidelity": ranked_full,
        "final_rows": final_rows,
        "summary_rows": summary_rows,
        "per_class_summary_rows": per_class_summary_rows,
        "selection_stage_seconds": float(selection_seconds),
        "final_stage_seconds": float(final_seconds),
        "total_workflow_seconds": float(total_workflow_seconds),
        "output_root": relative_output_path(RUN_ROOT, WORKSPACE_ROOT),
    }


## Module 11: Run switch and frozen comparison protocol

Keep the switch False while reviewing the notebook. Set it to True only when both yearly datasets and the training environment are ready. The shared data, runtime, search space, and compute settings should remain unchanged for fair optimizer comparison.


In [ ]:
RUN_CROSS_YEAR_DEHB = False

if RUN_CROSS_YEAR_DEHB:
    cross_year_dehb_result = run_cross_year_dehb_baseline()
else:
    print("RUN_CROSS_YEAR_DEHB is False.")
    print("Direction:", CROSS_YEAR_TAG)
    print(
        f"DEHB HPO: {DEHB_ITERATIONS} iterations x {DEHB_UNITS_PER_ITERATION} "
        f"= {DEHB_HPO_EPOCH_UNITS} epoch units"
    )
    print(
        f"Expected evaluations: {EXPECTED_HPO_EVALUATIONS} total, "
        f"{EXPECTED_FULL_FIDELITY_EVALUATIONS} at {DEHB_MAX_EPOCHS} epochs"
    )
    print(
        f"Final evaluation: {len(FINAL_EVALUATION_SEEDS)} x {FINAL_EPOCHS} "
        f"= {FINAL_EVALUATION_EPOCH_UNITS} epoch units, reported separately"
    )
    print("Fitness: source year validation mAP50-95 only")
    print("Target year test is not used until the DEHB winner is frozen.")
    print(
        f"Runtime: workers={WORKERS}, cache='{CACHE_MODE}', "
        f"batch={BATCH_SIZE}, imgsz={IMAGE_SIZE}, patience={PATIENCE}"
    )
    print("Year2021 source rule: subsample only source train to target train size.")
    print("Set RUN_CROSS_YEAR_DEHB = True when ready.")


## Frozen comparison protocol

Keep the pretrained YOLOv8m model, image size 640, batch size 16, workers 8, disk cache, patience 0, SGD optimizer, fixed augmentation settings, split seeds, five parameter search space, and source validation mAP50-95 objective unchanged across optimizer comparisons.

DEHB uses two 20/40 epoch iterations for exactly 320 HPO epoch units. Only successful 40 epoch evaluations are eligible to become the HPO winner. The target year test split is never used during HPO. Final evaluation uses seeds 42, 123, and 999 at 40 epochs each and is reported separately as 120 epoch units.
